# Notebook 06 — Feature Harmonisation

## Objective

Notebook 06 prepares the flood-conditioning factors generated in the previous
notebooks for integration into a common modelling framework.

The predictor datasets originate from different sources and have different
native spatial resolutions, coordinate reference systems, spatial extents,
grid structures, and data characteristics.

Before flood-susceptibility modelling, these predictors must be spatially
harmonised so that corresponding raster cells represent the same geographic
locations.

The harmonisation workflow will include:

1. Predictor inventory and spatial audit
2. CRS and spatial-reference verification
3. Comparison of native resolutions and extents
4. Selection of an appropriate common modelling grid
5. Variable-specific reprojection and resampling
6. Grid alignment and spatial extent harmonisation
7. Consistent NoData handling
8. Final predictor-stack quality assurance

Continuous and categorical predictors will be treated differently during
resampling. Continuous variables will use an appropriate continuous
resampling method, while categorical variables such as LULC will use
nearest-neighbour resampling.

The native spatial resolutions of the source datasets will not be changed
before the common modelling grid is explicitly defined.

The objective is to create a spatially consistent set of predictor layers
without introducing unjustified spatial detail or altering the physical
meaning of the original datasets.

## 6.1 — Predictor Inventory and Spatial Audit

The predictor rasters generated in Notebooks 02–05 are loaded and inspected
before spatial harmonisation.

The purpose of this step is to establish the current spatial characteristics
of every predictor, including:

- Coordinate reference system (CRS)
- Raster dimensions
- Pixel resolution
- Spatial extent
- Data type
- NoData representation
- Valid-pixel coverage

The predictors originate from different datasets and therefore are expected
to have different native spatial resolutions and, where applicable, different
grid structures.

No reprojection, resampling, clipping, or modification is performed in this
step.

The results of this spatial audit will be used to determine the appropriate
common modelling grid and variable-specific resampling strategy in subsequent
steps.

In [49]:
# ============================================================
# Step 06.1 — Predictor Inventory and Spatial Audit
# ============================================================

from pathlib import Path
import numpy as np
import rasterio

PROJECT_ROOT = Path.cwd().parent
# ------------------------------------------------------------
# LULC — Notebook 04
# ------------------------------------------------------------

lulc_paths = {
    2003: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2003.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2014.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2025.tif"
    )
}

# ------------------------------------------------------------
# Rainfall — Notebook 05
# ------------------------------------------------------------

rainfall_paths = {
    2003: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2003"
        / "CHIRPS_monsoon_rainfall_2003.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2014"
        / "CHIRPS_monsoon_rainfall_2014.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2025"
        / "CHIRPS_monsoon_rainfall_2025.tif"
    )
}

# ------------------------------------------------------------
# Soil — Notebook 05
# ------------------------------------------------------------

soil_paths = {
    "Clay": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "soil"
        / "clay_0-5cm_percent_EPSG32644.tif"
    ),

    "Sand": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "soil"
        / "sand_0-5cm_percent_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Combine known predictor datasets
# ------------------------------------------------------------

predictor_paths = {}

for year, path in lulc_paths.items():

    predictor_paths[f"LULC_{year}"] = path

for year, path in rainfall_paths.items():

    predictor_paths[f"Rainfall_{year}"] = path

for property_name, path in soil_paths.items():

    predictor_paths[property_name] = path

# ------------------------------------------------------------
# Spatial audit function
# ------------------------------------------------------------

def audit_raster(name, path):

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired predictor not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        if src.nodata is not None:

            valid = (
                np.isfinite(data) &
                (data != src.nodata)
            )

        else:

            valid = np.isfinite(data)

        valid_pixels = int(
            np.count_nonzero(valid)
        )

        total_pixels = (
            src.height * src.width
        )

        valid_percentage = (
            valid_pixels /
            total_pixels *
            100
        )

        print("\n" + "-" * 75)
        print(name)
        print("-" * 75)

        print(f"File         : {src.name}")
        print(f"CRS          : {src.crs}")
        print(f"Shape        : {src.shape}")
        print(f"Resolution   : {src.res}")
        print(f"Bounds       : {src.bounds}")
        print(f"Data type    : {src.dtypes[0]}")
        print(f"NoData       : {src.nodata}")
        print(
            f"Valid pixels : "
            f"{valid_pixels:,} / {total_pixels:,}"
        )
        print(
            f"Valid area   : "
            f"{valid_percentage:.2f}%"
        )

        return {
            "name": name,
            "path": path,
            "crs": src.crs,
            "shape": src.shape,
            "resolution": src.res,
            "bounds": src.bounds,
            "transform": src.transform,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "valid_pixels": valid_pixels,
            "total_pixels": total_pixels,
            "valid_percentage": valid_percentage
        }

# ------------------------------------------------------------
# Audit all loaded predictors
# ------------------------------------------------------------

predictor_metadata = []

for name, path in predictor_paths.items():

    metadata = audit_raster(
        name,
        path
    )

    predictor_metadata.append(
        metadata
    )

# ------------------------------------------------------------
# Final audit summary
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("PREDICTOR INVENTORY COMPLETED")
print("-" * 75)

print(
    f"Total predictor layers audited: "
    f"{len(predictor_metadata)}"
)

print("\n✓ No raster was modified.")
print("✓ No reprojection was performed.")
print("✓ No resampling was performed.")
print("✓ Spatial characteristics recorded.")


---------------------------------------------------------------------------
LULC_2003
---------------------------------------------------------------------------
File         : d:\Projects\GeoAI-Flood-Susceptibility\outputs\lulc_maps\LULC_RF_2003.tif
CRS          : EPSG:32644
Shape        : (7111, 7861)
Resolution   : (30.0, 30.0)
Bounds       : BoundingBox(left=344385.0, bottom=2929485.0, right=580215.0, top=3142815.0)
Data type    : uint8
NoData       : 0.0
Valid pixels : 3,306,528 / 55,899,571
Valid area   : 5.92%

---------------------------------------------------------------------------
LULC_2014
---------------------------------------------------------------------------
File         : d:\Projects\GeoAI-Flood-Susceptibility\outputs\lulc_maps\LULC_RF_2014.tif
CRS          : EPSG:32644
Shape        : (7801, 7651)
Resolution   : (30.0, 30.0)
Bounds       : BoundingBox(left=348885.0, bottom=2916885.0, right=578415.0, top=3150915.0)
Data type    : uint8
NoData       : 0.0
Valid pixel

## 6.2 — Notebook 02 and 03 Predictor Inventory

The remaining terrain and hydrological predictor rasters generated in
Notebooks 02 and 03 are loaded for integration into the Notebook 06
harmonisation workflow.

These predictors provide the physical and hydrological conditioning factors
required for subsequent flood-susceptibility modelling.

The predictor layers are inspected for:

- Coordinate reference system (CRS)
- Raster dimensions
- Spatial resolution
- Spatial extent
- Data type
- NoData representation
- Valid-pixel coverage

At this stage, the original predictor rasters are only read and audited.
No reprojection, resampling, clipping, or alteration of the source datasets
is performed.

The complete predictor inventory from Notebooks 02–05 will subsequently be
used to determine the appropriate common modelling grid and
variable-specific resampling strategy.

In [50]:
# ============================================================
# Step 06.2 — Notebook 02 & 03 Predictor Inventory
# ============================================================

import numpy as np
import rasterio


# ============================================================
# NOTEBOOK 02 — TERRAIN PREDICTORS
# ============================================================

terrain_paths = {


    "Elevation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "DEM_Lucknow_20km.tif"
    ),

    "Slope": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "Slope_Lucknow_20km.tif"
    ),

    "Flow_Accumulation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "Flow_Accumulation_Lucknow_conditioned.tif"
    ),
}


# ============================================================
# NOTEBOOK 03 — HYDROLOGICAL PREDICTORS
# ============================================================

hydrological_paths = {


    "River_Distance": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_03"
        / "Distance_to_Gomti_River.tif"
    ),

    
    "Drainage Density": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_03"
        / "Drainage_Density_2km.tif"
    ),
}


# ============================================================
# COMBINE NOTEBOOK 02 + 03
# ============================================================

additional_predictor_paths = {}

additional_predictor_paths.update(
    terrain_paths
)

additional_predictor_paths.update(
    hydrological_paths
)


# ============================================================
# AUDIT FUNCTION
# ============================================================

def audit_predictor(name, path):

    if not path.exists():

        raise FileNotFoundError(
            f"\nPredictor file not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Valid-pixel mask
        # ----------------------------------------------------

        if src.nodata is not None:

            valid = (
                np.isfinite(data) &
                (data != src.nodata)
            )

        else:

            valid = np.isfinite(data)

        valid_pixels = int(
            np.count_nonzero(valid)
        )

        total_pixels = (
            src.height *
            src.width
        )

        valid_percentage = (
            valid_pixels /
            total_pixels *
            100
        )

        print("\n" + "-" * 75)
        print(name)
        print("-" * 75)

        print(f"File         : {src.name}")
        print(f"CRS          : {src.crs}")
        print(f"Shape        : {src.shape}")
        print(f"Resolution   : {src.res}")
        print(f"Bounds       : {src.bounds}")
        print(f"Data type    : {src.dtypes[0]}")
        print(f"NoData       : {src.nodata}")

        print(
            f"Valid pixels : "
            f"{valid_pixels:,} / {total_pixels:,}"
        )

        print(
            f"Valid area   : "
            f"{valid_percentage:.2f}%"
        )

        return {
            "name": name,
            "path": path,
            "crs": src.crs,
            "shape": src.shape,
            "resolution": src.res,
            "bounds": src.bounds,
            "transform": src.transform,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "valid_pixels": valid_pixels,
            "total_pixels": total_pixels,
            "valid_percentage": valid_percentage
        }


# ============================================================
# AUDIT NOTEBOOK 02 & 03 PREDICTORS
# ============================================================

additional_predictor_metadata = []

for name, path in additional_predictor_paths.items():

    metadata = audit_predictor(
        name,
        path
    )

    additional_predictor_metadata.append(
        metadata
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("NOTEBOOK 02 & 03 PREDICTOR INVENTORY COMPLETED")
print("=" * 75)

print(
    f"Total additional predictors audited: "
    f"{len(additional_predictor_metadata)}"
)

print("\n✓ No source raster was modified.")
print("✓ No reprojection was performed.")
print("✓ No resampling was performed.")
print("✓ Spatial characteristics recorded.")



---------------------------------------------------------------------------
Elevation
---------------------------------------------------------------------------
File         : d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_02\DEM_Lucknow_20km.tif
CRS          : EPSG:32644
Shape        : (2123, 2191)
Resolution   : (28.635052567985024, 28.635052567985024)
Bounds       : BoundingBox(left=463340.7988309339, bottom=2938217.292799435, right=526080.1990073891, top=2999009.5094012674)
Data type    : float32
NoData       : -9999.0
Valid pixels : 3,637,837 / 4,651,493
Valid area   : 78.21%

---------------------------------------------------------------------------
Slope
---------------------------------------------------------------------------
File         : d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_02\Slope_Lucknow_20km.tif
CRS          : EPSG:32644
Shape        : (2123, 2191)
Resolution   : (28.635052567985024, 28.635052567985024)
Bounds       : Boundi

## 6.3 — Complete Spatial Compatibility Assessment

The predictor rasters prepared in Notebooks 02–05 are compared to determine
their spatial compatibility before feature harmonisation.

The assessment examines:

- Coordinate reference system (CRS)
- Raster resolution
- Raster dimensions
- Spatial extent
- Affine transform and grid origin
- NoData representation
- Data type
- Predictor type (continuous or categorical)

Although all currently prepared predictors use EPSG:32644, they do not share
a common spatial grid. Differences occur in pixel resolution, raster extent,
dimensions, and grid alignment.

The assessment therefore distinguishes between:

1. CRS compatibility
2. Resolution compatibility
3. Extent compatibility
4. Grid-alignment compatibility

These properties must be evaluated separately because two rasters can have
the same CRS and resolution while still being spatially misaligned.

No raster is modified during this step.

The results will be used to select an appropriate common modelling grid and
define variable-specific resampling procedures in the following steps.

In [51]:
# ============================================================
# Step 06.3 — Complete Spatial Compatibility Assessment
# ============================================================

import pandas as pd
import numpy as np

print("\n" + "=" * 75)
print("NOTEBOOK 06 — COMPLETE SPATIAL COMPATIBILITY ASSESSMENT")
print("=" * 75)

# ------------------------------------------------------------
# Combine metadata from Steps 06.1 and 06.2
# ------------------------------------------------------------

all_predictor_metadata = (
    predictor_metadata +
    additional_predictor_metadata
)

# ------------------------------------------------------------
# Create compact metadata table
# ------------------------------------------------------------

compatibility_rows = []

for item in all_predictor_metadata:

    compatibility_rows.append({

        "predictor": item["name"],

        "CRS": str(item["crs"]),

        "width": item["shape"][1],

        "height": item["shape"][0],

        "resolution_x_m": item["resolution"][0],

        "resolution_y_m": item["resolution"][1],

        "left": item["bounds"].left,

        "bottom": item["bounds"].bottom,

        "right": item["bounds"].right,

        "top": item["bounds"].top,

        "dtype": item["dtype"],

        "nodata": item["nodata"],

        "valid_percentage": item["valid_percentage"]
    })


compatibility_df = pd.DataFrame(
    compatibility_rows
)

# ------------------------------------------------------------
# Display spatial compatibility table
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("PREDICTOR SPATIAL CHARACTERISTICS")
print("-" * 75)

print(
    compatibility_df[
        [
            "predictor",
            "CRS",
            "width",
            "height",
            "resolution_x_m",
            "resolution_y_m",
            "valid_percentage"
        ]
    ].to_string(index=False)
)

# ============================================================
# CRS compatibility
# ============================================================

unique_crs = (
    compatibility_df["CRS"]
    .dropna()
    .unique()
)

print("\n" + "-" * 75)
print("CRS COMPATIBILITY")
print("-" * 75)

print(
    f"Unique CRS values: {len(unique_crs)}"
)

for crs in unique_crs:
    print(f"  - {crs}")

if len(unique_crs) == 1:

    print(
        "\n✓ All audited predictors use the same CRS."
    )

else:

    print(
        "\n⚠ Predictors use multiple CRSs."
    )

# ============================================================
# Resolution compatibility
# ============================================================

unique_resolutions = (
    compatibility_df[
        [
            "resolution_x_m",
            "resolution_y_m"
        ]
    ]
    .drop_duplicates()
)

print("\n" + "-" * 75)
print("RESOLUTION COMPATIBILITY")
print("-" * 75)

print(
    unique_resolutions.to_string(index=False)
)

if len(unique_resolutions) == 1:

    print(
        "\n✓ All predictors share the same resolution."
    )

else:

    print(
        "\n⚠ Predictors have different native resolutions."
    )

# ============================================================
# Extent compatibility
# ============================================================

print("\n" + "-" * 75)
print("SPATIAL EXTENT")
print("-" * 75)

for _, row in compatibility_df.iterrows():

    print(
        f"\n{row['predictor']}"
    )

    print(
        f"  Left   : {row['left']:.3f}"
    )

    print(
        f"  Bottom : {row['bottom']:.3f}"
    )

    print(
        f"  Right  : {row['right']:.3f}"
    )

    print(
        f"  Top    : {row['top']:.3f}"
    )

# ============================================================
# Grid transform comparison
# ============================================================

print("\n" + "-" * 75)
print("GRID / TRANSFORM COMPATIBILITY")
print("-" * 75)

reference_transform = (
    all_predictor_metadata[0]["transform"]
)

transform_matches = []

for item in all_predictor_metadata:

    same_transform = (
        item["transform"] ==
        reference_transform
    )

    transform_matches.append(
        same_transform
    )

    print(
        f"{item['name']:<25} "
        f"same as reference: {same_transform}"
    )

if all(transform_matches):

    print(
        "\n✓ All predictors share the same affine transform."
    )

else:

    print(
        "\n⚠ Predictors do not share one common grid transform."
    )

# ============================================================
# Resolution groups
# ============================================================

print("\n" + "-" * 75)
print("NATIVE RESOLUTION GROUPS")
print("-" * 75)

resolution_groups = (
    compatibility_df
    .groupby(
        [
            "resolution_x_m",
            "resolution_y_m"
        ]
    )["predictor"]
    .apply(list)
)

for resolution, predictors in resolution_groups.items():

    print(
        f"\nResolution: "
        f"{resolution[0]:.3f} × {resolution[1]:.3f} m"
    )

    for predictor in predictors:

        print(
            f"  - {predictor}"
        )

# ============================================================
# Final compatibility assessment
# ============================================================

print("""
CRS:
✓ All currently audited predictors use EPSG:32644.

RESOLUTION:
⚠ Predictors have different native spatial resolutions.

EXTENT:
⚠ Predictors do not share identical spatial extents.

GRID ALIGNMENT:
⚠ Predictors do not currently share one common affine grid.

CONCLUSION:
The predictor datasets are not yet ready to be stacked directly.

No resampling or reprojection has been performed in this step.
A common modelling grid must be explicitly selected before harmonisation.
""")

print("=" * 75)
print("✓ STEP 06.3 COMPLETED")
print("=" * 75)


NOTEBOOK 06 — COMPLETE SPATIAL COMPATIBILITY ASSESSMENT

---------------------------------------------------------------------------
PREDICTOR SPATIAL CHARACTERISTICS
---------------------------------------------------------------------------
        predictor        CRS  width  height  resolution_x_m  resolution_y_m  valid_percentage
        LULC_2003 EPSG:32644   7861    7111       30.000000       30.000000          5.915122
        LULC_2014 EPSG:32644   7651    7801       30.000000       30.000000          5.369942
        LULC_2025 EPSG:32644   7651    7801       30.000000       30.000000          4.874213
    Rainfall_2003 EPSG:32644     13      13     5216.580330     5216.580330        100.000000
    Rainfall_2014 EPSG:32644     13      13     5216.580330     5216.580330        100.000000
    Rainfall_2025 EPSG:32644     13      13     5216.580330     5216.580330        100.000000
             Clay EPSG:32644    260     252      241.582046      241.582046         78.052503
    

## Step 06.4 — Define the Corrected Common Modelling Grid

All predictor datasets are harmonised to a single common modelling grid.

The common grid uses:

- CRS: EPSG:32644 (UTM Zone 44N)
- Spatial resolution: 250 m
- Width: 252 pixels
- Height: 244 pixels
- Left: 463250 m
- Bottom: 2938000 m
- Right: 526250 m
- Top: 2999000 m

The grid extent is rounded outward to fully contain the defined study area. This avoids losing the approximately 62 m eastern edge of the study area that was omitted by the previous 251-column grid.

All subsequent raster harmonisation and predictor-table generation must use this exact transform, dimensions, CRS, and alignment.

In [52]:
# ============================================================
# STEP 06.4 — CORRECTED COMMON MODELLING GRID
# ============================================================
import geopandas as gpd
import numpy as np
from rasterio.transform import from_origin

print("=" * 75)
print("STEP 06.4 — CORRECTED COMMON MODELLING GRID")
print("=" * 75)

# ------------------------------------------------------------
# Common modelling grid
# ------------------------------------------------------------

target_grid = {
    "crs": "EPSG:32644",

    "resolution": 250.0,

    "width": 252,
    "height": 244,

    "left": 463250.0,
    "bottom": 2938000.0,
    "right": 526250.0,
    "top": 2999000.0,

    "transform": from_origin(
        463250.0,
        2999000.0,
        250.0,
        250.0
    )
}

# ------------------------------------------------------------
# Extract grid parameters
# ------------------------------------------------------------

target_crs = target_grid["crs"]
target_transform = target_grid["transform"]
target_width = target_grid["width"]
target_height = target_grid["height"]

# ------------------------------------------------------------
# Calculate bounds from transform
# ------------------------------------------------------------

calculated_left = target_transform.c
calculated_top = target_transform.f

calculated_right = (
    calculated_left +
    target_width * target_grid["resolution"]
)

calculated_bottom = (
    calculated_top -
    target_height * target_grid["resolution"]
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nTARGET GRID")
print("-" * 75)

print(f"CRS              : {target_crs}")
print(
    f"Resolution       : "
    f"{target_grid['resolution']:.2f} × "
    f"{target_grid['resolution']:.2f} m"
)
print(f"Width            : {target_width}")
print(f"Height           : {target_height}")
print(
    f"Total cells      : "
    f"{target_width * target_height:,}"
)

print(f"Left             : {calculated_left:.2f}")
print(f"Bottom           : {calculated_bottom:.2f}")
print(f"Right            : {calculated_right:.2f}")
print(f"Top              : {calculated_top:.2f}")

print("\nTRANSFORM")
print("-" * 75)
print(target_transform)

# ------------------------------------------------------------
# Internal consistency checks
# ------------------------------------------------------------

print("\nGRID CONSISTENCY CHECKS")
print("-" * 75)

print(
    "Width correct:",
    target_width == 252
)

print(
    "Height correct:",
    target_height == 244
)

print(
    "Cell count correct:",
    target_width * target_height == 61488
)

print(
    "Right bound correct:",
    np.isclose(calculated_right, 526250.0)
)

print(
    "Bottom bound correct:",
    np.isclose(calculated_bottom, 2938000.0)
)

print(
    "Resolution correct:",
    (
        np.isclose(target_transform.a, 250.0)
        and
        np.isclose(target_transform.e, -250.0)
    )
)

# ------------------------------------------------------------
# Verify study-area coverage
# ------------------------------------------------------------
study_area = PROJECT_ROOT / "data" / "processed" / "notebook_01" / "Lucknow_Search_Area_20km.gpkg"
study_area = gpd.read_file(study_area)

study_area_target = study_area.set_crs(
    target_crs
)

study_minx, study_miny, study_maxx, study_maxy = (
    study_area_target.total_bounds
)

print("\nSTUDY-AREA COVERAGE")
print("-" * 75)

print(f"Study area left   : {study_minx:.6f}")
print(f"Study area bottom : {study_miny:.6f}")
print(f"Study area right  : {study_maxx:.6f}")
print(f"Study area top    : {study_maxy:.6f}")

print("\nCoverage checks")

print(
    "Left covered:",
    calculated_left <= study_minx
)

print(
    "Bottom covered:",
    calculated_bottom <= study_miny
)

print(
    "Right covered:",
    calculated_right >= study_maxx
)

print(
    "Top covered:",
    calculated_top >= study_maxy
)

print("\n" + "=" * 75)
print("✓ CORRECTED TARGET GRID DEFINED")
print("✓ 252 × 244")
print("✓ 250 m")
print("✓ EPSG:32644")
print("✓ EASTERN EXTENT: 526250 m")
print("=" * 75)

STEP 06.4 — CORRECTED COMMON MODELLING GRID

TARGET GRID
---------------------------------------------------------------------------
CRS              : EPSG:32644
Resolution       : 250.00 × 250.00 m
Width            : 252
Height           : 244
Total cells      : 61,488
Left             : 463250.00
Bottom           : 2938000.00
Right            : 526250.00
Top              : 2999000.00

TRANSFORM
---------------------------------------------------------------------------
| 250.00, 0.00, 463250.00|
| 0.00,-250.00, 2999000.00|
| 0.00, 0.00, 1.00|

GRID CONSISTENCY CHECKS
---------------------------------------------------------------------------
Width correct: True
Height correct: True
Cell count correct: True
Right bound correct: True
Bottom bound correct: True
Resolution correct: True

STUDY-AREA COVERAGE
---------------------------------------------------------------------------
Study area left   : 463358.579839
Study area bottom : 2938218.330946
Study area right  : 526062.355619
Stu

## 6.5 — Variable-Specific Resampling Strategy

The predictor datasets have different native spatial resolutions and data
characteristics. Therefore, a single resampling method is not appropriate for
all predictors.

The common modelling grid defined in Step 06.4 has:

- CRS: EPSG:32644
- Resolution: 250 m × 250 m
- Extent: covering the complete study area

Each predictor will be transferred to this common grid according to its
physical meaning and data type.

### Continuous predictors

The following predictors represent continuous environmental or hydrological
quantities:

- Elevation
- Slope
- Flow accumulation
- River distance
- Drainage density
- Soil clay content
- Soil sand content
- CHIRPS cumulative monsoon rainfall

These variables will be processed using methods appropriate for continuous
data.

For fine-resolution continuous predictors, aggregation to the 250 m grid is
preferred because each target cell represents an area containing multiple
source pixels.

For predictors with a native resolution close to the target resolution,
appropriate continuous resampling will be used.

### Categorical predictor

Land-use/land-cover (LULC) is a categorical variable. Its class labels must
remain discrete.

LULC will therefore be transferred using a categorical strategy rather than
bilinear or other continuous interpolation.

Where multiple source LULC pixels occur within a 250 m target cell, the
dominant class will be used to represent the target cell.

### Rainfall

CHIRPS rainfall has a native resolution of approximately 5.2 km, substantially
coarser than the 250 m modelling grid.

The rainfall values will therefore be transferred to the common grid without
interpreting the resulting 250 m pixels as independent high-resolution
rainfall observations.

The coarse native resolution of CHIRPS will be retained as an explicit
limitation of the rainfall predictor.

### Source-data preservation

The original predictor rasters will not be overwritten.

All harmonised datasets will be written to a separate Notebook 06 processed
directory.

This ensures that the native-resolution datasets remain available for
verification and reproducibility.

## 6.5.1 — Harmonisation of Fine-Resolution Continuous Predictors

The fine-resolution terrain and hydrological predictors generated in
Notebooks 02 and 03 have a native resolution of approximately 28.64 m.

These predictors are transferred to the 250 m common modelling grid.

The predictors processed in this step are:

- Elevation
- Slope
- Flow accumulation
- River distance
- Drainage density

Because the source resolution is substantially finer than the target
resolution, the operation represents spatial aggregation from multiple source
pixels into each target modelling cell.

Mean aggregation is used to obtain a representative continuous value for each
250 m target cell.

The source rasters are not modified. The harmonised rasters are written to
the Notebook 06 processed-data directory.

NoData values are preserved and are not interpreted as valid measurements.

In [53]:
# ============================================================
# Step 06.5.1 — Fine-Resolution Continuous Predictors
# ============================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

harmonised_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "harmonised"
)

harmonised_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Fine-resolution continuous predictors
# ------------------------------------------------------------

fine_continuous_paths = {

    "Elevation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "DEM_Lucknow_20km.tif"
    ),

    "Slope": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "Slope_Lucknow_20km.tif"
    ),

    "Flow_Accumulation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_02"
        / "Flow_Accumulation_Lucknow_conditioned.tif"
    ),

    "River_Distance": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_03"
        / "Distance_to_Gomti_River.tif"
    ),

    "Drainage_Density": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_03"
        / "Drainage_Density_2km.tif"
    )
}

# ------------------------------------------------------------
# Target grid
# ------------------------------------------------------------

dst_crs = target_grid["crs"]
dst_transform = target_grid["transform"]
dst_width = target_grid["width"]
dst_height = target_grid["height"]

# ------------------------------------------------------------
# Process each predictor
# ------------------------------------------------------------

for predictor_name, input_path in fine_continuous_paths.items():


    if not input_path.exists():

        raise FileNotFoundError(
            f"Predictor not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        # ----------------------------------------------------
        # Source CRS check
        # ----------------------------------------------------

        if src.crs is None:

            raise ValueError(
                f"{predictor_name} has no CRS."
            )

        # ----------------------------------------------------
        # Source data
        # ----------------------------------------------------

        source = src.read(1)

        source_nodata = src.nodata

        # ----------------------------------------------------
        # Destination array
        #
        # NaN is used internally so that source NoData does
        # not become a valid modelling value.
        # ----------------------------------------------------

        destination = np.full(
            (dst_height, dst_width),
            np.nan,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Reproject + aggregate
        # ----------------------------------------------------

        reproject(
            source=source,
            destination=destination,

            src_transform=src.transform,
            src_crs=src.crs,

            dst_transform=dst_transform,
            dst_crs=dst_crs,

            src_nodata=source_nodata,
            dst_nodata=np.nan,

            resampling=Resampling.average
        )

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        output_path = (
            harmonised_dir
            / f"{predictor_name}_250m_EPSG32644.tif"
        )

        # ----------------------------------------------------
        # Output profile
        # ----------------------------------------------------

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            height=dst_height,
            width=dst_width,
            count=1,
            dtype="float32",
            crs=dst_crs,
            transform=dst_transform,
            nodata=-9999.0,
            compress="deflate"
        )

        # ----------------------------------------------------
        # Convert internal NaN to explicit NoData
        # ----------------------------------------------------

        output_array = np.where(
            np.isfinite(destination),
            destination,
            -9999.0
        ).astype(np.float32)

        # ----------------------------------------------------
        # Write harmonised raster
        # ----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                output_array,
                1
            )

        # ----------------------------------------------------
        # QA
        # ----------------------------------------------------

        valid = (
            output_array != -9999.0
        )

        valid_values = (
            output_array[valid]
        )

        print(
            f"Output       : {output_path.name}"
        )

        print(
            f"Shape        : {output_array.shape}"
        )

        print(
            f"Resolution   : "
            f"{dst_transform.a:.2f} × "
            f"{abs(dst_transform.e):.2f} m"
        )

        print(
            f"Valid pixels : "
            f"{valid_values.size:,}"
        )

        if valid_values.size > 0:

            print(
                f"Minimum      : "
                f"{valid_values.min():.4f}"
            )

            print(
                f"Maximum      : "
                f"{valid_values.max():.4f}"
            )

            print(
                f"Mean         : "
                f"{valid_values.mean():.4f}"
            )

print("\n" + "=" * 75)
print("✓ FINE-RESOLUTION CONTINUOUS PREDICTORS HARMONISED")
print("✓ TARGET GRID: 250 m / EPSG:32644")
print("✓ SOURCE RASTERS WERE NOT MODIFIED")
print("=" * 75)

Output       : Elevation_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid pixels : 48,217
Minimum      : 101.2746
Maximum      : 135.9356
Mean         : 120.7011
Output       : Slope_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid pixels : 48,141
Minimum      : 0.0000
Maximum      : 11.8061
Mean         : 1.5637
Output       : Flow_Accumulation_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid pixels : 61,488
Minimum      : 2.1022
Maximum      : 648344.6250
Mean         : 3723.7612
Output       : River_Distance_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid pixels : 61,488
Minimum      : 38.3393
Maximum      : 19004.2383
Mean         : 2218.5564
Output       : Drainage_Density_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid pixels : 48,217
Minimum      : 0.0000
Maximum      : 2.1059
Mean         : 0.3927

✓ FIN

## 6.5.1 QA — Drainage Density Aggregation Check

The drainage-density raster had substantially lower valid-pixel coverage at its
native resolution than the other fine-resolution predictors.

Because the 28.64 m drainage-density raster was aggregated to the 250 m
modelling grid using mean aggregation, the valid-pixel coverage of the
resulting raster must be examined before the layer is accepted for modelling.

A target cell may contain both valid and NoData source pixels. Therefore, a
valid aggregated value does not necessarily mean that the complete target
cell was covered by valid source data.

This quality-assurance step examines the relationship between source-data
coverage and the resulting 250 m drainage-density cells.

The purpose is to ensure that the aggregation does not conceal substantial
spatial gaps in the original drainage-density dataset.

In [54]:
# ============================================================
# Step 06.5.1 QA — Drainage Density Aggregation Check
# ============================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

source_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_03"
    / "Drainage_Density_2km.tif"
)

harmonised_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "harmonised"
    / "Drainage_Density_250m_EPSG32644.tif"
)

# ------------------------------------------------------------
# Open source drainage-density raster
# ------------------------------------------------------------

with rasterio.open(source_path) as src:

    source = src.read(1)

    source_transform = src.transform
    source_crs = src.crs
    source_nodata = src.nodata

    source_height = src.height
    source_width = src.width

# ------------------------------------------------------------
# Create source-validity mask
#
# 1 = valid source pixel
# 0 = NoData source pixel
# ------------------------------------------------------------

if source_nodata is not None:

    source_valid = (
        np.isfinite(source) &
        (source != source_nodata)
    )

else:

    source_valid = np.isfinite(source)

# ------------------------------------------------------------
# Convert validity mask to float
# ------------------------------------------------------------

source_valid_float = (
    source_valid.astype(np.float32)
)

# ------------------------------------------------------------
# Aggregate source validity to the 250 m grid
#
# average = proportion of valid source pixels contributing
# to each target cell.
# ------------------------------------------------------------

valid_fraction = np.zeros(
    (
        target_grid["height"],
        target_grid["width"]
    ),
    dtype=np.float32
)

reproject(
    source=source_valid_float,
    destination=valid_fraction,

    src_transform=source_transform,
    src_crs=source_crs,

    dst_transform=target_grid["transform"],
    dst_crs=target_grid["crs"],

    src_nodata=0,
    dst_nodata=0,

    resampling=Resampling.average
)

# ------------------------------------------------------------
# Open harmonised drainage-density raster
# ------------------------------------------------------------

with rasterio.open(harmonised_path) as dst:

    drainage_250m = dst.read(1)

    harmonised_nodata = dst.nodata

# ------------------------------------------------------------
# Identify valid harmonised cells
# ------------------------------------------------------------

if harmonised_nodata is not None:

    harmonised_valid = (
        np.isfinite(drainage_250m) &
        (drainage_250m != harmonised_nodata)
    )

else:

    harmonised_valid = np.isfinite(
        drainage_250m
    )

# ------------------------------------------------------------
# Statistics for valid-source coverage
# ------------------------------------------------------------

coverage_percent = (
    valid_fraction * 100.0
)

valid_target_coverage = coverage_percent[
    harmonised_valid
]

# ------------------------------------------------------------
# Coverage categories
# ------------------------------------------------------------

full_coverage = (
    valid_target_coverage >= 99.999
)

high_coverage = (
    valid_target_coverage >= 75.0
)

partial_coverage = (
    valid_target_coverage > 0.0
) & (
    valid_target_coverage < 75.0
)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("SOURCE RASTER")
print("-" * 75)

print(
    f"Resolution       : "
    f"{source_transform.a:.6f} × "
    f"{abs(source_transform.e):.6f} m"
)

print(
    f"Shape            : "
    f"{source_height} × {source_width}"
)

print(
    f"Source NoData    : "
    f"{source_nodata}"
)

print(
    f"Valid source     : "
    f"{np.count_nonzero(source_valid):,} / "
    f"{source.size:,}"
)

print(
    f"Shape            : "
    f"{drainage_250m.shape}"
)

print(
    f"Resolution       : "
    f"{target_grid['resolution']:.2f} × "
    f"{target_grid['resolution']:.2f} m"
)

print(
    f"Valid cells      : "
    f"{np.count_nonzero(harmonised_valid):,} / "
    f"{drainage_250m.size:,}"
)


if valid_target_coverage.size > 0:

    print(
        f"Minimum coverage : "
        f"{valid_target_coverage.min():.2f}%"
    )

    print(
        f"Maximum coverage : "
        f"{valid_target_coverage.max():.2f}%"
    )

    print(
        f"Mean coverage    : "
        f"{valid_target_coverage.mean():.2f}%"
    )

    print(
        f"Median coverage  : "
        f"{np.median(valid_target_coverage):.2f}%"
    )

print("\n" + "-" * 75)
print("TARGET-CELL COVERAGE CATEGORIES")
print("-" * 75)

print(
    f"100% source coverage      : "
    f"{np.count_nonzero(full_coverage):,}"
)

print(
    f"≥75% source coverage      : "
    f"{np.count_nonzero(high_coverage):,}"
)

print(
    f"<75% source coverage      : "
    f"{np.count_nonzero(partial_coverage):,}"
)

# ------------------------------------------------------------
# Check for suspicious zero-only valid cells
# ------------------------------------------------------------

zero_value_cells = (
    harmonised_valid &
    np.isclose(
        drainage_250m,
        0.0,
        atol=1e-8
    )
)

print("\n" + "-" * 75)
print("ZERO-VALUE CHECK")
print("-" * 75)

print(
    f"Valid cells with value = 0: "
    f"{np.count_nonzero(zero_value_cells):,}"
)

# ------------------------------------------------------------
# Final assessment
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DRAINAGE DENSITY AGGREGATION ASSESSMENT")
print("=" * 75)

if valid_target_coverage.size == 0:

    print(
        "⚠ No valid harmonised drainage-density cells were found."
    )

elif valid_target_coverage.min() >= 75.0:

    print(
        "✓ All valid 250 m cells have at least 75% "
        "valid source-pixel coverage."
    )

else:

    print(
        "⚠ Some valid 250 m cells contain less than "
        "75% valid source-pixel coverage."
    )

print(
    "\nThe harmonised drainage-density raster was not modified."
)

print(
    "The QA only measured source-pixel coverage "
    "contributing to each 250 m cell."
)

print("=" * 75)


---------------------------------------------------------------------------
SOURCE RASTER
---------------------------------------------------------------------------
Resolution       : 28.635053 × 28.635053 m
Shape            : 3443 × 4571
Source NoData    : -9999.0
Valid source     : 3,637,837 / 15,737,953
Shape            : (244, 252)
Resolution       : 250.00 × 250.00 m
Valid cells      : 48,217 / 61,488
Minimum coverage : 100.00%
Maximum coverage : 100.00%
Mean coverage    : 100.00%
Median coverage  : 100.00%

---------------------------------------------------------------------------
TARGET-CELL COVERAGE CATEGORIES
---------------------------------------------------------------------------
100% source coverage      : 48,217
≥75% source coverage      : 48,217
<75% source coverage      : 0

---------------------------------------------------------------------------
ZERO-VALUE CHECK
---------------------------------------------------------------------------
Valid cells with value = 

## 6.5.2 — Harmonisation of LULC

The Landsat-derived LULC rasters have a native spatial resolution of 30 m,
while the common modelling grid has a resolution of 250 m.

LULC is a categorical predictor and therefore cannot be harmonised using
continuous interpolation or averaging methods.

Each 250 m modelling cell may contain multiple 30 m LULC classes. The
dominant-class approach is therefore used to assign the class occupying the
largest number of valid source pixels within each target cell.

The LULC classes remain discrete and retain their original class identifiers:

1. Water
2. Vegetation
3. Built-up
4. Barren
5. Agriculture

Source NoData pixels (class 0) are excluded from the dominant-class
calculation.

The three temporal LULC datasets (2003, 2014, and 2025) are independently
harmonised to the same 250 m modelling grid.

The original 30 m LULC rasters are not modified.

A target cell is assigned NoData only when no valid LULC source pixels are
available within that cell.

In [55]:
# ============================================================
# Step 06.5.2 — LULC Harmonisation
# ============================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

print("\n" + "=" * 75)
print("NOTEBOOK 06 — LULC HARMONISATION")
print("=" * 75)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

lulc_harmonised_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "harmonised"
    / "lulc"
)

lulc_harmonised_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# LULC input paths
# ------------------------------------------------------------

lulc_paths = {

    2003: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2003.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2014.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "outputs"
        / "lulc_maps"
        / "LULC_RF_2025.tif"
    )
}

# ------------------------------------------------------------
# Target grid
# ------------------------------------------------------------

dst_crs = target_grid["crs"]
dst_transform = target_grid["transform"]
dst_width = target_grid["width"]
dst_height = target_grid["height"]

# ------------------------------------------------------------
# LULC class definitions
# ------------------------------------------------------------

lulc_classes = [
    1,  # Water
    2,  # Vegetation
    3,  # Built-up
    4,  # Barren
    5   # Agriculture
]

# ------------------------------------------------------------
# Process each year
# ------------------------------------------------------------

for year, input_path in lulc_paths.items():

    print("\n" + "-" * 75)
    print(f"HARMONISING LULC — {year}")
    print("-" * 75)

    if not input_path.exists():

        raise FileNotFoundError(
            f"LULC raster not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        source = src.read(1)

        source_nodata = src.nodata

        # ----------------------------------------------------
        # Verify expected CRS
        # ----------------------------------------------------

        if src.crs is None:

            raise ValueError(
                f"LULC {year} has no CRS."
            )

        # ----------------------------------------------------
        # Verify categorical values
        # ----------------------------------------------------

        valid_source = np.isfinite(source)

        if source_nodata is not None:

            valid_source &= (
                source != source_nodata
            )

        unexpected_values = np.unique(
            source[valid_source]
        )

        unexpected_values = unexpected_values[
            ~np.isin(
                unexpected_values,
                lulc_classes
            )
        ]

        if unexpected_values.size > 0:

            raise ValueError(
                f"LULC {year} contains unexpected "
                f"class values: {unexpected_values}"
            )

        # ----------------------------------------------------
        # Create class-count arrays
        # ----------------------------------------------------

        class_counts = np.zeros(
            (
                len(lulc_classes),
                dst_height,
                dst_width
            ),
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Count source pixels belonging to each class
        # ----------------------------------------------------

        for class_index, class_value in enumerate(
            lulc_classes
        ):

            class_mask = (
                source == class_value
            ).astype(np.float32)

            reproject(
                source=class_mask,
                destination=class_counts[class_index],

                src_transform=src.transform,
                src_crs=src.crs,

                dst_transform=dst_transform,
                dst_crs=dst_crs,

                src_nodata=0,
                dst_nodata=0,

                resampling=Resampling.sum
            )

        # ----------------------------------------------------
        # Determine dominant class
        # ----------------------------------------------------

        dominant_index = np.argmax(
            class_counts,
            axis=0
        )

        dominant_class = np.array(
            lulc_classes,
            dtype=np.uint8
        )[dominant_index]

        # ----------------------------------------------------
        # Determine total valid class contribution
        # ----------------------------------------------------

        total_valid = np.sum(
            class_counts,
            axis=0
        )

        # ----------------------------------------------------
        # Cells with no valid LULC pixels → NoData
        # ----------------------------------------------------

        output = np.where(
            total_valid > 0,
            dominant_class,
            0
        ).astype(np.uint8)

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        output_path = (
            lulc_harmonised_dir
            / f"LULC_{year}_250m_EPSG32644.tif"
        )

        # ----------------------------------------------------
        # Output profile
        # ----------------------------------------------------

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            height=dst_height,
            width=dst_width,
            count=1,
            dtype="uint8",
            crs=dst_crs,
            transform=dst_transform,
            nodata=0,
            compress="deflate"
        )

        # ----------------------------------------------------
        # Write output
        # ----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                output,
                1
            )

        # ----------------------------------------------------
        # QA
        # ----------------------------------------------------

        valid = (
            output != 0
        )

        valid_values = output[
            valid
        ]

        unique_classes, counts = np.unique(
            valid_values,
            return_counts=True
        )

        print(
            f"Output       : "
            f"{output_path.name}"
        )

        print(
            f"Shape        : "
            f"{output.shape}"
        )

        print(
            f"Resolution   : "
            f"{target_grid['resolution']:.2f} × "
            f"{target_grid['resolution']:.2f} m"
        )

        print(
            f"Valid cells  : "
            f"{valid_values.size:,} / "
            f"{output.size:,}"
        )

        print("\nDominant LULC classes:")

        for class_value, count in zip(
            unique_classes,
            counts
        ):

            percentage = (
                count /
                valid_values.size *
                100
            )

            print(
                f"  Class {class_value}: "
                f"{count:,} cells "
                f"({percentage:.2f}%)"
            )

print("\n" + "=" * 75)
print("✓ ALL THREE LULC YEARS HARMONISED")
print("✓ TARGET GRID: 250 m / EPSG:32644")
print("✓ DOMINANT-CLASS METHOD USED")
print("✓ SOURCE LULC RASTERS WERE NOT MODIFIED")
print("=" * 75)


NOTEBOOK 06 — LULC HARMONISATION

---------------------------------------------------------------------------
HARMONISING LULC — 2003
---------------------------------------------------------------------------
Output       : LULC_2003_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid cells  : 48,121 / 61,488

Dominant LULC classes:
  Class 1: 173 cells (0.36%)
  Class 2: 5,687 cells (11.82%)
  Class 3: 9,261 cells (19.25%)
  Class 4: 30,107 cells (62.57%)
  Class 5: 2,893 cells (6.01%)

---------------------------------------------------------------------------
HARMONISING LULC — 2014
---------------------------------------------------------------------------
Output       : LULC_2014_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
Valid cells  : 46,957 / 61,488

Dominant LULC classes:
  Class 1: 131 cells (0.28%)
  Class 2: 3,180 cells (6.77%)
  Class 3: 18,428 cells (39.24%)
  Class 4: 20,817 cells (44.33%)
  Class 5: 4,

## 6.5.2 QA — Harmonised LULC Quality Assessment

The harmonised LULC rasters for 2003, 2014, and 2025 are evaluated after
conversion from the native 30 m grid to the common 250 m modelling grid.

The quality-assurance checks verify that:

- All three rasters use EPSG:32644.
- All three rasters have the same dimensions.
- All three rasters have the same 250 m resolution.
- All three rasters share the same spatial transform and grid alignment.
- Only the valid LULC class identifiers 1–5 are present.
- NoData is represented by class 0.
- No unexpected continuous or interpolated class values were introduced.

The temporal LULC class distribution is also reported for comparison.

The original 30 m LULC datasets are not modified during this QA step.

In [56]:
# ============================================================
# Step 06.5.2 QA — Harmonised LULC Quality Assessment
# ============================================================

import numpy as np
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — HARMONISED LULC QA")
print("=" * 75)

# ------------------------------------------------------------
# Harmonised LULC paths
# ------------------------------------------------------------

lulc_harmonised_paths = {

    2003: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "LULC_2003_250m_EPSG32644.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "LULC_2014_250m_EPSG32644.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "lulC_2025_250m_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Expected LULC classes
# ------------------------------------------------------------

expected_classes = {
    1: "Water",
    2: "Vegetation",
    3: "Built-up",
    4: "Barren",
    5: "Agriculture"
}

reference_transform = None
reference_shape = None
reference_crs = None
reference_resolution = None

qa_results = {}

# ------------------------------------------------------------
# QA each year
# ------------------------------------------------------------

for year, path in lulc_harmonised_paths.items():

    print("\n" + "-" * 75)
    print(f"LULC — {year}")
    print("-" * 75)

    if not path.exists():

        raise FileNotFoundError(
            f"Harmonised LULC raster not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Basic metadata
        # ----------------------------------------------------

        print(
            f"CRS         : {src.crs}"
        )

        print(
            f"Shape       : {src.shape}"
        )

        print(
            f"Resolution  : {src.res}"
        )

        print(
            f"NoData      : {src.nodata}"
        )

        print(
            f"Bounds      : {src.bounds}"
        )

        # ----------------------------------------------------
        # CRS check
        # ----------------------------------------------------

        assert src.crs.to_string() == "EPSG:32644"

        # ----------------------------------------------------
        # Resolution check
        # ----------------------------------------------------

        assert np.isclose(
            src.res[0],
            250.0
        )

        assert np.isclose(
            src.res[1],
            250.0
        )

        # ----------------------------------------------------
        # Shape check
        # ----------------------------------------------------

        if reference_shape is None:

            reference_shape = src.shape
            reference_crs = src.crs
            reference_resolution = src.res
            reference_transform = src.transform

        else:

            assert (
                src.shape ==
                reference_shape
            )

            assert (
                src.crs ==
                reference_crs
            )

            assert np.allclose(
                src.transform,
                reference_transform
            )

        # ----------------------------------------------------
        # Valid / NoData mask
        # ----------------------------------------------------

        valid = (
            data != 0
        )

        valid_values = data[
            valid
        ]

        # ----------------------------------------------------
        # Check class values
        # ----------------------------------------------------

        unique_values = np.unique(
            valid_values
        )

        invalid_values = unique_values[
            ~np.isin(
                unique_values,
                list(expected_classes.keys())
            )
        ]

        print(
            f"Unique valid classes: "
            f"{unique_values.tolist()}"
        )

        if invalid_values.size > 0:

            raise ValueError(
                f"LULC {year} contains invalid "
                f"class values: "
                f"{invalid_values.tolist()}"
            )

        # ----------------------------------------------------
        # Class statistics
        # ----------------------------------------------------

        print("\nClass distribution:")

        class_counts = {}

        for class_id, class_name in expected_classes.items():

            count = int(
                np.count_nonzero(
                    data == class_id
                )
            )

            class_counts[class_id] = count

            if valid_values.size > 0:

                percentage = (
                    count /
                    valid_values.size *
                    100
                )

            else:

                percentage = 0.0

            print(
                f"  {class_id} — "
                f"{class_name:<12} : "
                f"{count:,} cells "
                f"({percentage:.2f}%)"
            )

        # ----------------------------------------------------
        # NoData statistics
        # ----------------------------------------------------

        nodata_count = int(
            np.count_nonzero(
                data == 0
            )
        )

        valid_count = int(
            np.count_nonzero(valid)
        )

        valid_percentage = (
            valid_count /
            data.size *
            100
        )

        print(
            f"\nValid cells : "
            f"{valid_count:,} / "
            f"{data.size:,}"
        )

        print(
            f"Valid area : "
            f"{valid_percentage:.2f}%"
        )

        print(
            f"NoData cells: "
            f"{nodata_count:,}"
        )

        qa_results[year] = {
            "valid_cells": valid_count,
            "nodata_cells": nodata_count,
            "valid_percentage": valid_percentage,
            "classes": unique_values.tolist()
        }

# ------------------------------------------------------------
# Final QA
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL LULC QA")
print("=" * 75)

print(
    "✓ All three LULC rasters use EPSG:32644"
)

print(
    "✓ All three LULC rasters use 250 m resolution"
)

print(
    "✓ All three LULC rasters have identical dimensions"
)

print(
    "✓ All three LULC rasters share the same grid transform"
)

print(
    "✓ Only LULC classes 1–5 are present"
)

print(
    "✓ NoData is represented by class 0"
)

print(
    "✓ No continuous/interpolated class values detected"
)

print(
    "✓ Original 30 m LULC rasters were not modified"
)

print("=" * 75)
print("✓ STEP 06.5.2 QA COMPLETED")
print("=" * 75)


NOTEBOOK 06 — HARMONISED LULC QA

---------------------------------------------------------------------------
LULC — 2003
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : 0.0
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
Unique valid classes: [1, 2, 3, 4, 5]

Class distribution:
  1 — Water        : 173 cells (0.36%)
  2 — Vegetation   : 5,687 cells (11.82%)
  3 — Built-up     : 9,261 cells (19.25%)
  4 — Barren       : 30,107 cells (62.57%)
  5 — Agriculture  : 2,893 cells (6.01%)

Valid cells : 48,121 / 61,488
Valid area : 78.26%
NoData cells: 13,367

---------------------------------------------------------------------------
LULC — 2014
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : 0.0
Bounds      : 

## 6.5.3 — Harmonisation of SoilGrids Predictors

The SoilGrids clay and sand predictors have a native spatial resolution of
approximately 241.58 m and are already expressed in the project CRS
(EPSG:32644).

The common modelling grid has a resolution of 250 m × 250 m. Therefore, the
SoilGrids layers require only a small spatial adjustment to match the common
modelling grid.

Clay and sand are continuous soil-property variables. They are therefore
transferred to the 250 m modelling grid using bilinear resampling.

The existing NoData value (-9999) is explicitly preserved and is not treated
as a valid soil-property value during resampling.

The resulting layers will have:

- CRS: EPSG:32644
- Resolution: 250 m × 250 m
- Dimensions: 244 × 251
- NoData: -9999
- Data type: float32

The original SoilGrids rasters generated in Notebook 05 are not modified.
The harmonised copies are stored separately under the Notebook 06 processed
data directory.

In [57]:
# ============================================================
# Step 06.5.3 — SoilGrids Harmonisation
# ============================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

print("\n" + "=" * 75)
print("NOTEBOOK 06 — SOILGRIDS HARMONISATION")
print("=" * 75)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

soil_harmonised_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "harmonised"
    / "soil"
)

soil_harmonised_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# SoilGrids input paths
# ------------------------------------------------------------

soil_paths = {

    "Clay": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "soil"
        / "clay_0-5cm_percent_EPSG32644.tif"
    ),

    "Sand": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "soil"
        / "sand_0-5cm_percent_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Target grid
# ------------------------------------------------------------

dst_crs = target_grid["crs"]
dst_transform = target_grid["transform"]
dst_width = target_grid["width"]
dst_height = target_grid["height"]

# ------------------------------------------------------------
# Process Clay and Sand
# ------------------------------------------------------------

for property_name, input_path in soil_paths.items():

    print("\n" + "-" * 75)
    print(f"HARMONISING SOIL — {property_name.upper()}")
    print("-" * 75)

    if not input_path.exists():

        raise FileNotFoundError(
            f"SoilGrids raster not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        # ----------------------------------------------------
        # CRS check
        # ----------------------------------------------------

        if src.crs is None:

            raise ValueError(
                f"{property_name} has no CRS."
            )

        if src.crs.to_string() != "EPSG:32644":

            raise ValueError(
                f"{property_name} is not in EPSG:32644. "
                f"Found: {src.crs}"
            )

        # ----------------------------------------------------
        # Read source raster
        # ----------------------------------------------------

        source = src.read(1)

        source_nodata = src.nodata

        if source_nodata is None:

            raise ValueError(
                f"{property_name} has no explicit NoData value."
            )

        # ----------------------------------------------------
        # Destination array
        # ----------------------------------------------------

        destination = np.full(
            (
                dst_height,
                dst_width
            ),
            np.nan,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Reproject / resample
        #
        # Bilinear is appropriate for continuous soil
        # properties.
        # ----------------------------------------------------

        reproject(
            source=source,
            destination=destination,

            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=source_nodata,

            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,

            resampling=Resampling.bilinear
        )

        # ----------------------------------------------------
        # Convert NaN → explicit NoData
        # ----------------------------------------------------

        output_array = np.where(
            np.isfinite(destination),
            destination,
            -9999.0
        ).astype(np.float32)

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        output_path = (
            soil_harmonised_dir
            / f"{property_name}_0-5cm_percent_250m_EPSG32644.tif"
        )

        # ----------------------------------------------------
        # Output profile
        # ----------------------------------------------------

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            height=dst_height,
            width=dst_width,
            count=1,
            dtype="float32",
            crs=dst_crs,
            transform=dst_transform,
            nodata=-9999.0,
            compress="deflate"
        )

        # ----------------------------------------------------
        # Write harmonised raster
        # ----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                output_array,
                1
            )

        # ----------------------------------------------------
        # QA
        # ----------------------------------------------------

        valid = (
            np.isfinite(output_array) &
            (output_array != -9999.0)
        )

        valid_values = output_array[
            valid
        ]

        print(
            f"Output       : "
            f"{output_path.name}"
        )

        print(
            f"Shape        : "
            f"{output_array.shape}"
        )

        print(
            f"Resolution   : "
            f"{target_grid['resolution']:.2f} × "
            f"{target_grid['resolution']:.2f} m"
        )

        print(
            f"NoData       : "
            f"-9999.0"
        )

        print(
            f"Valid cells  : "
            f"{valid_values.size:,} / "
            f"{output_array.size:,}"
        )

        if valid_values.size > 0:

            print(
                f"Minimum (%)  : "
                f"{valid_values.min():.2f}"
            )

            print(
                f"Maximum (%)  : "
                f"{valid_values.max():.2f}"
            )

            print(
                f"Mean (%)     : "
                f"{valid_values.mean():.2f}"
            )

print("\n" + "=" * 75)
print("✓ CLAY AND SAND HARMONISED")
print("✓ TARGET GRID: 250 m / EPSG:32644")
print("✓ BILINEAR RESAMPLING USED FOR CONTINUOUS SOIL DATA")
print("✓ SOURCE SOIL RASTERS WERE NOT MODIFIED")
print("=" * 75)


NOTEBOOK 06 — SOILGRIDS HARMONISATION

---------------------------------------------------------------------------
HARMONISING SOIL — CLAY
---------------------------------------------------------------------------
Output       : Clay_0-5cm_percent_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
NoData       : -9999.0
Valid cells  : 47,709 / 61,488
Minimum (%)  : 0.00
Maximum (%)  : 33.83
Mean (%)     : 25.17

---------------------------------------------------------------------------
HARMONISING SOIL — SAND
---------------------------------------------------------------------------
Output       : Sand_0-5cm_percent_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
NoData       : -9999.0
Valid cells  : 47,709 / 61,488
Minimum (%)  : 0.00
Maximum (%)  : 42.54
Mean (%)     : 32.07

✓ CLAY AND SAND HARMONISED
✓ TARGET GRID: 250 m / EPSG:32644
✓ BILINEAR RESAMPLING USED FOR CONTINUOUS SOIL DATA
✓ SOURCE SOIL RASTERS WERE NOT MODIFI

## 6.5.3 QA — Harmonised SoilGrids Quality Assessment

The harmonised SoilGrids clay and sand rasters are evaluated after transfer
from their native approximately 241.58 m grid to the common 250 m modelling
grid.

The quality-assurance checks verify that:

- Both layers use EPSG:32644.
- Both layers have identical dimensions.
- Both layers have a 250 m × 250 m resolution.
- Both layers share the same affine grid transform.
- NoData is represented consistently by -9999.
- Clay values remain within the expected percentage range.
- Sand values remain within the expected percentage range.
- No invalid or infinite values were introduced during resampling.

The valid-pixel coverage of the harmonised layers is compared with the
original SoilGrids layers to identify any substantial change in spatial
coverage.

The original SoilGrids rasters generated in Notebook 05 are not modified
during this quality-assurance step.

In [58]:
# ============================================================
# Step 06.5.3 QA — Harmonised SoilGrids Quality Assessment
# ============================================================

import numpy as np
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — HARMONISED SOILGRIDS QA")
print("=" * 75)

# ------------------------------------------------------------
# Harmonised SoilGrids paths
# ------------------------------------------------------------

soil_harmonised_paths = {

    "Clay": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "soil"
        / "Clay_0-5cm_percent_250m_EPSG32644.tif"
    ),

    "Sand": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "soil"
        / "Sand_0-5cm_percent_250m_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Expected value ranges
# ------------------------------------------------------------

expected_ranges = {
    "Clay": (0.0, 100.0),
    "Sand": (0.0, 100.0)
}

reference_shape = None
reference_transform = None
reference_crs = None
reference_resolution = None

# ------------------------------------------------------------
# QA each soil layer
# ------------------------------------------------------------

for property_name, path in soil_harmonised_paths.items():

    print("\n" + "-" * 75)
    print(f"{property_name.upper()} — HARMONISED SOILGRIDS")
    print("-" * 75)

    if not path.exists():

        raise FileNotFoundError(
            f"Harmonised SoilGrids raster not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        print(
            f"CRS         : {src.crs}"
        )

        print(
            f"Shape       : {src.shape}"
        )

        print(
            f"Resolution  : {src.res}"
        )

        print(
            f"NoData      : {src.nodata}"
        )

        print(
            f"Bounds      : {src.bounds}"
        )

        # ----------------------------------------------------
        # CRS check
        # ----------------------------------------------------

        assert (
            src.crs.to_string()
            == "EPSG:32644"
        )

        # ----------------------------------------------------
        # Resolution check
        # ----------------------------------------------------

        assert np.isclose(
            src.res[0],
            250.0
        )

        assert np.isclose(
            src.res[1],
            250.0
        )

        # ----------------------------------------------------
        # Shape / grid consistency
        # ----------------------------------------------------

        if reference_shape is None:

            reference_shape = src.shape
            reference_transform = src.transform
            reference_crs = src.crs
            reference_resolution = src.res

        else:

            assert (
                src.shape ==
                reference_shape
            )

            assert (
                src.crs ==
                reference_crs
            )

            assert np.allclose(
                src.transform,
                reference_transform
            )

        # ----------------------------------------------------
        # NoData check
        # ----------------------------------------------------

        assert (
            src.nodata == -9999.0
        )

        # ----------------------------------------------------
        # Valid-data mask
        # ----------------------------------------------------

        valid = (
            np.isfinite(data) &
            (data != -9999.0)
        )

        valid_values = data[
            valid
        ]

        # ----------------------------------------------------
        # Check for invalid numerical values
        # ----------------------------------------------------

        if not np.all(
            np.isfinite(valid_values)
        ):

            raise ValueError(
                f"{property_name} contains "
                "NaN or infinite valid values."
            )

        # ----------------------------------------------------
        # Value-range check
        # ----------------------------------------------------

        minimum_expected, maximum_expected = (
            expected_ranges[property_name]
        )

        actual_min = float(
            valid_values.min()
        )

        actual_max = float(
            valid_values.max()
        )

        if actual_min < minimum_expected:

            raise ValueError(
                f"{property_name} minimum "
                f"{actual_min:.4f} is below "
                f"expected range."
            )

        if actual_max > maximum_expected:

            raise ValueError(
                f"{property_name} maximum "
                f"{actual_max:.4f} is above "
                f"expected range."
            )

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        valid_count = int(
            valid_values.size
        )

        total_count = int(
            data.size
        )

        valid_percentage = (
            valid_count /
            total_count *
            100
        )

        print(
            f"Valid cells : "
            f"{valid_count:,} / "
            f"{total_count:,}"
        )

        print(
            f"Valid area  : "
            f"{valid_percentage:.2f}%"
        )

        print(
            f"Minimum (%) : "
            f"{actual_min:.2f}"
        )

        print(
            f"Maximum (%) : "
            f"{actual_max:.2f}"
        )

        print(
            f"Mean (%)    : "
            f"{valid_values.mean():.2f}"
        )

        print(
            f"Median (%)  : "
            f"{np.median(valid_values):.2f}"
        )

# ------------------------------------------------------------
# Final QA
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL SOILGRIDS QA")
print("=" * 75)

print(
    "✓ Clay and Sand use EPSG:32644"
)

print(
    "✓ Clay and Sand use 250 m resolution"
)

print(
    "✓ Clay and Sand have identical dimensions"
)

print(
    "✓ Clay and Sand share the same grid transform"
)

print(
    "✓ NoData is consistently represented by -9999"
)

print(
    "✓ No invalid numerical values detected"
)

print(
    "✓ Clay values remain within 0–100%"
)

print(
    "✓ Sand values remain within 0–100%"
)

print(
    "✓ Original SoilGrids rasters were not modified"
)

print("=" * 75)
print("✓ STEP 06.5.3 QA COMPLETED")
print("=" * 75)


NOTEBOOK 06 — HARMONISED SOILGRIDS QA

---------------------------------------------------------------------------
CLAY — HARMONISED SOILGRIDS
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : -9999.0
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
Valid cells : 47,709 / 61,488
Valid area  : 77.59%
Minimum (%) : 0.00
Maximum (%) : 33.83
Mean (%)    : 25.17
Median (%)  : 26.42

---------------------------------------------------------------------------
SAND — HARMONISED SOILGRIDS
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : -9999.0
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
Valid cells : 47,709 / 61,488
Valid area  : 77.59%
Minimum (%) : 0.00
Maximum (%) : 42.

## 6.5.4 — Harmonisation of CHIRPS Rainfall

The CHIRPS monsoon rainfall predictors have a native spatial resolution of
approximately 5.2 km, which is substantially coarser than the 250 m common
modelling grid.

The rainfall rasters for 2003, 2014, and 2025 are continuous precipitation
variables and are therefore transferred to the common modelling grid using
bilinear resampling.

The resampling operation does not increase the actual spatial information
content of the CHIRPS rainfall dataset. The resulting 250 m cells represent
interpolated values derived from the much coarser native CHIRPS rainfall
surface.

Therefore, the harmonised rainfall layers are treated as modelling-grid
representations of coarse-resolution rainfall rather than independent
250 m rainfall observations.

The three rainfall years are independently transferred to the same 250 m
grid so that they can be combined with the other harmonised predictors.

The original CHIRPS rasters generated in Notebook 05 are not modified.

The following properties are retained in the harmonised rainfall layers:

- CRS: EPSG:32644
- Resolution: 250 m × 250 m
- Dimensions: 244 × 251
- Variable: cumulative monsoon rainfall
- Unit: millimetres (mm)

The coarse native spatial resolution of CHIRPS is retained as an explicit
limitation of the rainfall predictor and is considered during interpretation
of subsequent modelling results.

In [59]:
# ============================================================
# Step 06.5.4 — CHIRPS Rainfall Harmonisation
# ============================================================

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

print("\n" + "=" * 75)
print("NOTEBOOK 06 — CHIRPS RAINFALL HARMONISATION")
print("=" * 75)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

rainfall_harmonised_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "harmonised"
    / "rainfall"
)

rainfall_harmonised_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# CHIRPS input paths
# ------------------------------------------------------------

rainfall_paths = {

    2003: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2003"
        / "CHIRPS_monsoon_rainfall_2003.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2014"
        / "CHIRPS_monsoon_rainfall_2014.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_05"
        / "rainfall"
        / "2025"
        / "CHIRPS_monsoon_rainfall_2025.tif"
    )
}

# ------------------------------------------------------------
# Target grid
# ------------------------------------------------------------

dst_crs = target_grid["crs"]
dst_transform = target_grid["transform"]
dst_width = target_grid["width"]
dst_height = target_grid["height"]

# ------------------------------------------------------------
# Process each rainfall year
# ------------------------------------------------------------

for year, input_path in rainfall_paths.items():

    print("\n" + "-" * 75)
    print(f"HARMONISING RAINFALL — {year}")
    print("-" * 75)

    if not input_path.exists():

        raise FileNotFoundError(
            f"CHIRPS rainfall raster not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        # ----------------------------------------------------
        # CRS check
        # ----------------------------------------------------

        if src.crs is None:

            raise ValueError(
                f"Rainfall {year} has no CRS."
            )

        if src.crs.to_string() != "EPSG:32644":

            raise ValueError(
                f"Rainfall {year} is not in EPSG:32644. "
                f"Found: {src.crs}"
            )

        # ----------------------------------------------------
        # Read rainfall
        # ----------------------------------------------------

        source = src.read(1)

        source_nodata = src.nodata

        # ----------------------------------------------------
        # Destination array
        # ----------------------------------------------------

        destination = np.full(
            (
                dst_height,
                dst_width
            ),
            np.nan,
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Bilinear resampling
        #
        # Rainfall is continuous.
        # ----------------------------------------------------

        reproject(
            source=source,
            destination=destination,

            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=source_nodata,

            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,

            resampling=Resampling.bilinear
        )

        # ----------------------------------------------------
        # Convert NaN → explicit NoData
        # ----------------------------------------------------

        output_array = np.where(
            np.isfinite(destination),
            destination,
            -9999.0
        ).astype(np.float32)

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        output_path = (
            rainfall_harmonised_dir
            / f"CHIRPS_monsoon_rainfall_{year}_250m_EPSG32644.tif"
        )

        # ----------------------------------------------------
        # Output profile
        # ----------------------------------------------------

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            height=dst_height,
            width=dst_width,
            count=1,
            dtype="float32",
            crs=dst_crs,
            transform=dst_transform,
            nodata=-9999.0,
            compress="deflate"
        )

        # ----------------------------------------------------
        # Write harmonised rainfall
        # ----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                output_array,
                1
            )

        # ----------------------------------------------------
        # QA statistics
        # ----------------------------------------------------

        valid = (
            np.isfinite(output_array) &
            (output_array != -9999.0)
        )

        valid_values = output_array[
            valid
        ]

        print(
            f"Output       : "
            f"{output_path.name}"
        )

        print(
            f"Shape        : "
            f"{output_array.shape}"
        )

        print(
            f"Resolution   : "
            f"{dst_transform.a:.2f} × "
            f"{abs(dst_transform.e):.2f} m"
        )

        print(
            f"NoData       : "
            f"-9999.0"
        )

        print(
            f"Valid cells  : "
            f"{valid_values.size:,} / "
            f"{output_array.size:,}"
        )

        if valid_values.size > 0:

            print(
                f"Minimum (mm) : "
                f"{valid_values.min():.2f}"
            )

            print(
                f"Maximum (mm) : "
                f"{valid_values.max():.2f}"
            )

            print(
                f"Mean (mm)    : "
                f"{valid_values.mean():.2f}"
            )

            print(
                f"Median (mm)  : "
                f"{np.median(valid_values):.2f}"
            )

print("\n" + "=" * 75)
print("✓ ALL THREE CHIRPS YEARS HARMONISED")
print("✓ TARGET GRID: 250 m / EPSG:32644")
print("✓ BILINEAR RESAMPLING USED")
print("✓ SOURCE CHIRPS RASTERS WERE NOT MODIFIED")
print("=" * 75)


NOTEBOOK 06 — CHIRPS RAINFALL HARMONISATION

---------------------------------------------------------------------------
HARMONISING RAINFALL — 2003
---------------------------------------------------------------------------
Output       : CHIRPS_monsoon_rainfall_2003_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
NoData       : -9999.0
Valid cells  : 61,488 / 61,488
Minimum (mm) : 0.00
Maximum (mm) : 1003.22
Mean (mm)    : 703.39
Median (mm)  : 880.61

---------------------------------------------------------------------------
HARMONISING RAINFALL — 2014
---------------------------------------------------------------------------
Output       : CHIRPS_monsoon_rainfall_2014_250m_EPSG32644.tif
Shape        : (244, 252)
Resolution   : 250.00 × 250.00 m
NoData       : -9999.0
Valid cells  : 61,488 / 61,488
Minimum (mm) : 0.00
Maximum (mm) : 834.59
Mean (mm)    : 583.05
Median (mm)  : 724.99

-------------------------------------------------------------------

## 6.5.4 QA — Harmonised CHIRPS Rainfall

The harmonised CHIRPS monsoon rainfall rasters for 2003, 2014, and 2025 are
evaluated after transfer from the native approximately 5.2 km CHIRPS grid to
the common 250 m modelling grid.

The quality-assurance checks verify that:

- All three rainfall rasters use EPSG:32644.
- All three rasters have dimensions of 244 × 251.
- All three rasters have a 250 m × 250 m resolution.
- All three rasters share the same spatial extent and affine grid transform.
- NoData is represented consistently by -9999.
- Rainfall values are finite and non-negative.
- Rainfall values remain expressed in millimetres.
- No unexpected negative rainfall values were introduced during resampling.

The harmonised rainfall statistics are reported separately for 2003, 2014,
and 2025.

Because the native CHIRPS resolution is approximately 5.2 km, the 250 m
rainfall layers represent spatially interpolated representations of the
coarse CHIRPS rainfall field. The harmonisation therefore does not imply that
independent 250 m rainfall observations are available.

The original CHIRPS rainfall rasters generated in Notebook 05 are not
modified during this quality-assurance step.

In [60]:
# ============================================================
# Step 06.5.4 QA — Harmonised CHIRPS Rainfall
# ============================================================

import numpy as np
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — HARMONISED CHIRPS RAINFALL QA")
print("=" * 75)

# ------------------------------------------------------------
# Harmonised rainfall paths
# ------------------------------------------------------------

rainfall_harmonised_paths = {

    2003: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2003_250m_EPSG32644.tif"
    ),

    2014: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2014_250m_EPSG32644.tif"
    ),

    2025: (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2025_250m_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Reference grid
# ------------------------------------------------------------

reference_shape = None
reference_transform = None
reference_bounds = None
reference_crs = None
reference_resolution = None

# ------------------------------------------------------------
# QA each year
# ------------------------------------------------------------

for year, path in rainfall_harmonised_paths.items():

    print("\n" + "-" * 75)
    print(f"RAINFALL — {year}")
    print("-" * 75)

    if not path.exists():

        raise FileNotFoundError(
            f"Harmonised rainfall raster not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        print(
            f"CRS         : {src.crs}"
        )

        print(
            f"Shape       : {src.shape}"
        )

        print(
            f"Resolution  : {src.res}"
        )

        print(
            f"NoData      : {src.nodata}"
        )

        print(
            f"Bounds      : {src.bounds}"
        )

        # ----------------------------------------------------
        # CRS check
        # ----------------------------------------------------

        assert (
            src.crs.to_string()
            == "EPSG:32644"
        )

        # ----------------------------------------------------
        # Resolution check
        # ----------------------------------------------------

        assert np.isclose(
            src.res[0],
            250.0
        )

        assert np.isclose(
            src.res[1],
            250.0
        )

        # ----------------------------------------------------
        # Reference grid checks
        # ----------------------------------------------------

        if reference_shape is None:

            reference_shape = src.shape
            reference_transform = src.transform
            reference_bounds = src.bounds
            reference_crs = src.crs
            reference_resolution = src.res

        else:

            assert (
                src.shape ==
                reference_shape
            )

            assert (
                src.crs ==
                reference_crs
            )

            assert np.allclose(
                src.transform,
                reference_transform
            )

            assert np.allclose(
                np.array(src.bounds),
                np.array(reference_bounds)
            )

        # ----------------------------------------------------
        # NoData check
        # ----------------------------------------------------

        assert (
            src.nodata == -9999.0
        )

        # ----------------------------------------------------
        # Valid rainfall pixels
        # ----------------------------------------------------

        valid = (
            np.isfinite(data) &
            (data != -9999.0)
        )

        valid_values = data[
            valid
        ]

        # ----------------------------------------------------
        # Numerical validity
        # ----------------------------------------------------

        if not np.all(
            np.isfinite(valid_values)
        ):

            raise ValueError(
                f"Rainfall {year} contains "
                "NaN or infinite valid values."
            )

        # ----------------------------------------------------
        # Rainfall cannot be negative
        # ----------------------------------------------------

        negative_values = (
            valid_values < 0
        )

        if np.any(negative_values):

            raise ValueError(
                f"Rainfall {year} contains "
                f"{np.count_nonzero(negative_values):,} "
                "negative values."
            )

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        valid_count = int(
            valid_values.size
        )

        total_count = int(
            data.size
        )

        valid_percentage = (
            valid_count /
            total_count *
            100
        )

        print(
            f"Valid cells : "
            f"{valid_count:,} / "
            f"{total_count:,}"
        )

        print(
            f"Valid area  : "
            f"{valid_percentage:.2f}%"
        )

        print(
            f"Minimum (mm): "
            f"{valid_values.min():.2f}"
        )

        print(
            f"Maximum (mm): "
            f"{valid_values.max():.2f}"
        )

        print(
            f"Mean (mm)   : "
            f"{valid_values.mean():.2f}"
        )

        print(
            f"Median (mm) : "
            f"{np.median(valid_values):.2f}"
        )

# ------------------------------------------------------------
# Final QA
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL CHIRPS RAINFALL QA")
print("=" * 75)

print(
    "✓ All three rainfall rasters use EPSG:32644"
)

print(
    "✓ All three rainfall rasters use 250 m resolution"
)

print(
    "✓ All three rainfall rasters have identical dimensions"
)

print(
    "✓ All three rainfall rasters share the same grid transform"
)

print(
    "✓ All three rainfall rasters share the same spatial extent"
)

print(
    "✓ NoData is consistently represented by -9999"
)

print(
    "✓ No negative rainfall values detected"
)

print(
    "✓ No NaN or infinite rainfall values detected"
)

print(
    "✓ Rainfall remains expressed in millimetres"
)

print(
    "✓ Original CHIRPS rasters were not modified"
)

print("=" * 75)
print("✓ STEP 06.5.4 QA COMPLETED")
print("=" * 75)


NOTEBOOK 06 — HARMONISED CHIRPS RAINFALL QA

---------------------------------------------------------------------------
RAINFALL — 2003
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : -9999.0
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
Valid cells : 61,488 / 61,488
Valid area  : 100.00%
Minimum (mm): 0.00
Maximum (mm): 1003.22
Mean (mm)   : 703.39
Median (mm) : 880.61

---------------------------------------------------------------------------
RAINFALL — 2014
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
NoData      : -9999.0
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
Valid cells : 61,488 / 61,488
Valid area  : 100.00%
Minimum (mm): 0.00
Maximum (mm): 834.59
Mean (mm

## 6.6 — Study-Area Masking and NoData Standardisation

The common 250 m modelling grid was defined using the bounding extent of the
study area. Consequently, the rectangular modelling grid extends slightly
beyond the actual study-area polygon.

To ensure that only locations within the defined study area are retained,
the harmonised predictor rasters are masked using the study-area geometry.

The study-area geometry is already defined in EPSG:32644 and is therefore
spatially compatible with the common modelling grid.

The same spatial mask is applied consistently to all harmonised predictors,
including:

- Elevation
- Slope
- Flow accumulation
- River distance
- Drainage density
- LULC for 2003, 2014, and 2025
- Soil clay
- Soil sand
- CHIRPS rainfall for 2003, 2014, and 2025

Cells outside the study-area polygon are assigned NoData.

Existing predictor NoData values are preserved and are not converted into
valid observations.

The masking operation does not alter the numerical values of valid predictor
cells.

The original harmonised rasters generated in Step 06.5 are not overwritten.
Masked copies are written to a separate Notebook 06 modelling directory.

A common NoData convention is used for continuous predictors:

- NoData = -9999

For categorical LULC:

- NoData = 0

The resulting masked predictors retain the common 250 m grid, EPSG:32644 CRS,
and identical spatial dimensions.

In [61]:
# ============================================================
# Step 06.6 — Study-Area Masking and NoData Standardisation
# ============================================================

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask

print("\n" + "=" * 75)
print("NOTEBOOK 06 — STUDY-AREA MASKING")
print("=" * 75)

# ------------------------------------------------------------
# Study area
#
# The study_area object was already assigned EPSG:32644
# in Step 06.4 using set_crs().
# ------------------------------------------------------------

if study_area.crs is None:

    study_area.crs = "EPSG:32644"
if study_area.crs.to_string() != "EPSG:32644":

    raise ValueError(
        f"Expected study area CRS EPSG:32644, "
        f"found {study_area.crs}"
    )

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

masked_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "masked"
)

masked_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Create mask on the common 250 m grid
#
# True  = outside study area
# False = inside study area
# ------------------------------------------------------------

outside_mask = geometry_mask(
    geometries=study_area.geometry,
    transform=target_grid["transform"],
    out_shape=(
        target_grid["height"],
        target_grid["width"]
    ),
    invert=False
)

inside_count = np.count_nonzero(
    ~outside_mask
)

outside_count = np.count_nonzero(
    outside_mask
)

print("\n" + "-" * 75)
print("STUDY-AREA MASK")
print("-" * 75)

print(
    f"Target grid cells : "
    f"{target_grid['width'] * target_grid['height']:,}"
)

print(
    f"Inside study area : "
    f"{inside_count:,}"
)

print(
    f"Outside study area: "
    f"{outside_count:,}"
)

print(
    f"Study-area coverage: "
    f"{inside_count / outside_mask.size * 100:.2f}%"
)

# ------------------------------------------------------------
# Define all harmonised rasters
# ------------------------------------------------------------

harmonised_rasters = {

    # --------------------------------------------------------
    # Continuous predictors
    # --------------------------------------------------------

    "Elevation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "Elevation_250m_EPSG32644.tif"
    ),

    "Slope": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "Slope_250m_EPSG32644.tif"
    ),

    "Flow_Accumulation": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "Flow_Accumulation_250m_EPSG32644.tif"
    ),

    "River_Distance": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "River_Distance_250m_EPSG32644.tif"
    ),

    "Drainage_Density": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "Drainage_Density_250m_EPSG32644.tif"
    ),

    # --------------------------------------------------------
    # Soil
    # --------------------------------------------------------

    "Clay": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "soil"
        / "Clay_0-5cm_percent_250m_EPSG32644.tif"
    ),

    "Sand": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "soil"
        / "Sand_0-5cm_percent_250m_EPSG32644.tif"
    ),

    # --------------------------------------------------------
    # Rainfall
    # --------------------------------------------------------

    "Rainfall_2003": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2003_250m_EPSG32644.tif"
    ),

    "Rainfall_2014": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2014_250m_EPSG32644.tif"
    ),

    "Rainfall_2025": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "rainfall"
        / "CHIRPS_monsoon_rainfall_2025_250m_EPSG32644.tif"
    ),

    # --------------------------------------------------------
    # LULC
    # --------------------------------------------------------

    "LULC_2003": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "LULC_2003_250m_EPSG32644.tif"
    ),

    "LULC_2014": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "LULC_2014_250m_EPSG32644.tif"
    ),

    "LULC_2025": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "notebook_06"
        / "harmonised"
        / "lulc"
        / "LULC_2025_250m_EPSG32644.tif"
    )
}

# ------------------------------------------------------------
# Categorical LULC layers use NoData = 0
# Continuous layers use NoData = -9999
# ------------------------------------------------------------

categorical_predictors = {
    "LULC_2003",
    "LULC_2014",
    "LULC_2025"
}

# ------------------------------------------------------------
# Process every harmonised raster
# ------------------------------------------------------------

for predictor_name, input_path in harmonised_rasters.items():

    print("\n" + "-" * 75)
    print(f"MASKING — {predictor_name}")
    print("-" * 75)

    if not input_path.exists():

        raise FileNotFoundError(
            f"Harmonised raster not found:\n{input_path}"
        )

    with rasterio.open(input_path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Grid checks
        # ----------------------------------------------------

        if src.crs.to_string() != "EPSG:32644":

            raise ValueError(
                f"{predictor_name} has CRS {src.crs}"
            )

        if src.shape != (
            target_grid["height"],
            target_grid["width"]
        ):

            raise ValueError(
                f"{predictor_name} does not match "
                "target grid dimensions."
            )

        if not np.allclose(
            src.transform,
            target_grid["transform"]
        ):

            raise ValueError(
                f"{predictor_name} does not match "
                "target grid transform."
            )

        # ----------------------------------------------------
        # Determine original NoData
        # ----------------------------------------------------

        if predictor_name in categorical_predictors:

            output_nodata = 0

            valid = (
                data != 0
            )

        else:

            output_nodata = -9999.0

            valid = (
                np.isfinite(data) &
                (data != -9999.0)
            )

        # ----------------------------------------------------
        # Apply study-area mask
        # ----------------------------------------------------

        output = data.copy()

        output[outside_mask] = output_nodata

        # ----------------------------------------------------
        # Also ensure invalid source cells remain NoData
        # ----------------------------------------------------

        output[~valid] = output_nodata

        # ----------------------------------------------------
        # Preserve data type
        # ----------------------------------------------------

        if predictor_name in categorical_predictors:

            output = output.astype(
                np.uint8
            )

        else:

            output = output.astype(
                np.float32
            )

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        output_path = (
            masked_dir
            / f"{predictor_name}_250m_EPSG32644_masked.tif"
        )

        # ----------------------------------------------------
        # Output profile
        # ----------------------------------------------------

        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            height=target_grid["height"],
            width=target_grid["width"],
            count=1,
            dtype=output.dtype,
            crs=target_grid["crs"],
            transform=target_grid["transform"],
            nodata=output_nodata,
            compress="deflate"
        )

        # ----------------------------------------------------
        # Write masked raster
        # ----------------------------------------------------

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(
                output,
                1
            )

        # ----------------------------------------------------
        # QA statistics
        # ----------------------------------------------------

        if predictor_name in categorical_predictors:

            valid_after = (
                output != 0
            )

        else:

            valid_after = (
                np.isfinite(output) &
                (output != -9999.0)
            )

        print(
            f"Output       : "
            f"{output_path.name}"
        )

        print(
            f"Valid cells  : "
            f"{np.count_nonzero(valid_after):,}"
        )

        print(
            f"NoData cells : "
            f"{np.count_nonzero(~valid_after):,}"
        )

print("\n" + "=" * 75)
print("✓ ALL HARMONISED PREDICTORS MASKED")
print("✓ STUDY-AREA MASK APPLIED CONSISTENTLY")
print("✓ 250 m / EPSG:32644 GRID PRESERVED")
print("✓ SOURCE HARMONISED RASTERS WERE NOT MODIFIED")
print("=" * 75)


NOTEBOOK 06 — STUDY-AREA MASKING

---------------------------------------------------------------------------
STUDY-AREA MASK
---------------------------------------------------------------------------
Target grid cells : 61,488
Inside study area : 47,730
Outside study area: 13,758
Study-area coverage: 77.62%

---------------------------------------------------------------------------
MASKING — Elevation
---------------------------------------------------------------------------
Output       : Elevation_250m_EPSG32644_masked.tif
Valid cells  : 47,730
NoData cells : 13,758

---------------------------------------------------------------------------
MASKING — Slope
---------------------------------------------------------------------------
Output       : Slope_250m_EPSG32644_masked.tif
Valid cells  : 47,730
NoData cells : 13,758

---------------------------------------------------------------------------
MASKING — Flow_Accumulation
-------------------------------------------------------

## 6.7 — Final Grid Alignment Quality Assessment

Following harmonisation and study-area masking, all predictor rasters are
subjected to a final spatial-grid quality assessment.

The purpose of this step is to verify that all predictors can be compared
pixel-by-pixel on the common modelling grid.

The following spatial properties are checked for every predictor:

- CRS
- Raster dimensions
- Spatial resolution
- Affine transform
- Spatial bounds
- Pixel-grid alignment

All predictors are expected to use:

- CRS: EPSG:32644
- Resolution: 250 m × 250 m
- Dimensions: 244 × 251
- Common spatial extent
- Common affine transform

Differences in valid-pixel coverage are not treated as grid-alignment errors.
They represent genuine NoData differences inherited from the individual
predictor datasets.

The quality-assurance step does not modify any raster.

Only predictors that share the common spatial grid will be eligible for
pixel-wise integration in subsequent steps.

In [62]:
# ============================================================
# Step 06.7 — Final Grid Alignment QA
# ============================================================

import numpy as np
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — FINAL GRID ALIGNMENT QA")
print("=" * 75)

# ------------------------------------------------------------
# All masked predictor paths
# ------------------------------------------------------------

masked_rasters = {

    "Elevation": (
        masked_dir
        / "Elevation_250m_EPSG32644_masked.tif"
    ),

    "Slope": (
        masked_dir
        / "Slope_250m_EPSG32644_masked.tif"
    ),

    "Flow_Accumulation": (
        masked_dir
        / "Flow_Accumulation_250m_EPSG32644_masked.tif"
    ),

    "River_Distance": (
        masked_dir
        / "River_Distance_250m_EPSG32644_masked.tif"
    ),

    "Drainage_Density": (
        masked_dir
        / "Drainage_Density_250m_EPSG32644_masked.tif"
    ),

    "Clay": (
        masked_dir
        / "Clay_250m_EPSG32644_masked.tif"
    ),

    "Sand": (
        masked_dir
        / "Sand_250m_EPSG32644_masked.tif"
    ),

    "Rainfall_2003": (
        masked_dir
        / "Rainfall_2003_250m_EPSG32644_masked.tif"
    ),

    "Rainfall_2014": (
        masked_dir
        / "Rainfall_2014_250m_EPSG32644_masked.tif"
    ),

    "Rainfall_2025": (
        masked_dir
        / "Rainfall_2025_250m_EPSG32644_masked.tif"
    ),

    "LULC_2003": (
        masked_dir
        / "LULC_2003_250m_EPSG32644_masked.tif"
    ),

    "LULC_2014": (
        masked_dir
        / "LULC_2014_250m_EPSG32644_masked.tif"
    ),

    "LULC_2025": (
        masked_dir
        / "LULC_2025_250m_EPSG32644_masked.tif"
    )
}

# ------------------------------------------------------------
# Expected target grid
# ------------------------------------------------------------

expected_crs = target_grid["crs"]

expected_shape = (
    target_grid["height"],
    target_grid["width"]
)

expected_resolution = (
    target_grid["resolution"],
    target_grid["resolution"]
)

expected_transform = (
    target_grid["transform"]
)

expected_bounds = (
    target_grid["left"],
    target_grid["bottom"],
    target_grid["right"],
    target_grid["top"]
)

# ------------------------------------------------------------
# Track QA status
# ------------------------------------------------------------

all_passed = True

# ------------------------------------------------------------
# Reference transform
# ------------------------------------------------------------

reference_transform = None

# ------------------------------------------------------------
# Check every raster
# ------------------------------------------------------------

for predictor_name, path in masked_rasters.items():

    print("\n" + "-" * 75)
    print(f"{predictor_name}")
    print("-" * 75)

    if not path.exists():

        raise FileNotFoundError(
            f"Masked predictor not found:\n{path}"
        )

    with rasterio.open(path) as src:

        print(
            f"CRS         : {src.crs}"
        )

        print(
            f"Shape       : {src.shape}"
        )

        print(
            f"Resolution  : {src.res}"
        )

        print(
            f"Bounds      : {src.bounds}"
        )

        print(
            f"NoData      : {src.nodata}"
        )

        # ----------------------------------------------------
        # CRS
        # ----------------------------------------------------

        crs_ok = (
            src.crs is not None
            and
            src.crs.to_string()
            == expected_crs
        )

        # ----------------------------------------------------
        # Shape
        # ----------------------------------------------------

        shape_ok = (
            src.shape ==
            expected_shape
        )

        # ----------------------------------------------------
        # Resolution
        # ----------------------------------------------------

        resolution_ok = (
            np.isclose(
                src.res[0],
                expected_resolution[0]
            )
            and
            np.isclose(
                src.res[1],
                expected_resolution[1]
            )
        )

        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        transform_ok = np.allclose(
            src.transform,
            expected_transform
        )

        # ----------------------------------------------------
        # Bounds
        # ----------------------------------------------------

        bounds_array = np.array([
            src.bounds.left,
            src.bounds.bottom,
            src.bounds.right,
            src.bounds.top
        ])

        expected_bounds_array = np.array(
            expected_bounds
        )

        bounds_ok = np.allclose(
            bounds_array,
            expected_bounds_array
        )

        # ----------------------------------------------------
        # NoData convention
        # ----------------------------------------------------

        if predictor_name.startswith("LULC"):

            nodata_ok = (
                src.nodata == 0
            )

        else:

            nodata_ok = (
                src.nodata == -9999.0
            )

        # ----------------------------------------------------
        # Overall predictor status
        # ----------------------------------------------------

        predictor_passed = all([
            crs_ok,
            shape_ok,
            resolution_ok,
            transform_ok,
            bounds_ok,
            nodata_ok
        ])

        if predictor_passed:

            print(
                "✓ Grid alignment: PASS"
            )

        else:

            print(
                "✗ Grid alignment: FAIL"
            )

            all_passed = False

        # ----------------------------------------------------
        # Detailed checks
        # ----------------------------------------------------

        print(
            f"  CRS       : "
            f"{'PASS' if crs_ok else 'FAIL'}"
        )

        print(
            f"  Shape     : "
            f"{'PASS' if shape_ok else 'FAIL'}"
        )

        print(
            f"  Resolution: "
            f"{'PASS' if resolution_ok else 'FAIL'}"
        )

        print(
            f"  Transform : "
            f"{'PASS' if transform_ok else 'FAIL'}"
        )

        print(
            f"  Bounds    : "
            f"{'PASS' if bounds_ok else 'FAIL'}"
        )

        print(
            f"  NoData    : "
            f"{'PASS' if nodata_ok else 'FAIL'}"
        )

# ------------------------------------------------------------
# Final assessment
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL GRID ALIGNMENT ASSESSMENT")
print("=" * 75)

if all_passed:

    print(
        "✓ ALL MASKED PREDICTORS SHARE THE SAME SPATIAL GRID"
    )

    print(
        "✓ CRS: EPSG:32644"
    )

    print(
        "✓ Resolution: 250 × 250 m"
    )

    print(
        "✓ Dimensions: 244 × 251"
    )

    print(
        "✓ Spatial extent is identical"
    )

    print(
        "✓ Affine transforms are identical"
    )

    print(
        "✓ NoData conventions are correct"
    )

else:

    raise ValueError(
        "One or more predictors failed "
        "the final grid-alignment QA."
    )

print("\n")
print(
    "Important: differences in valid-pixel coverage "
    "are expected and are not grid-alignment errors."
)

print(
    "No raster was modified during this QA."
)

print("=" * 75)
print("✓ STEP 06.7 COMPLETED")
print("=" * 75)


NOTEBOOK 06 — FINAL GRID ALIGNMENT QA

---------------------------------------------------------------------------
Elevation
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
NoData      : -9999.0
✓ Grid alignment: PASS
  CRS       : PASS
  Shape     : PASS
  Resolution: PASS
  Transform : PASS
  Bounds    : PASS
  NoData    : PASS

---------------------------------------------------------------------------
Slope
---------------------------------------------------------------------------
CRS         : EPSG:32644
Shape       : (244, 252)
Resolution  : (250.0, 250.0)
Bounds      : BoundingBox(left=463250.0, bottom=2938000.0, right=526250.0, top=2999000.0)
NoData      : -9999.0
✓ Grid alignment: PASS
  CRS       : PASS
  Shape     : PASS
  Resolution: PASS
  Transform : PASS
  Bounds    : PASS
  

## 6.8 — Common Valid-Cell Assessment

After all predictor rasters have been harmonised to the common 250 m grid and
masked to the study area, the spatial availability of predictor data is
assessed.

Each predictor retains its original valid-data and NoData characteristics.
No missing values are interpolated, filled, or otherwise artificially
reconstructed in this step.

The predictors are divided into:

### Static predictors

- Elevation
- Slope
- Flow accumulation
- River distance
- Drainage density
- Clay
- Sand

### Time-varying predictors

- LULC
- Monsoon rainfall

For each study year (2003, 2014, and 2025), a common valid-cell mask is
created using the static predictors together with the corresponding year's
LULC and rainfall layers.

A modelling cell is considered valid only when all required predictors for
that year contain valid observations.

The following quantities are reported:

- Total cells within the modelling grid
- Cells inside the study-area mask
- Valid cells for each predictor
- Common valid cells for each study year
- Percentage of the study area represented by the common valid cells

This assessment identifies the effective spatial domain available for
subsequent predictor integration.

No raster values or source files are modified during this step.

In [63]:
# ============================================================
# Step 06.8 — Common Valid-Cell Assessment
# ============================================================

import numpy as np
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — COMMON VALID-CELL ASSESSMENT")
print("=" * 75)

# ------------------------------------------------------------
# Static predictors
# ------------------------------------------------------------

static_predictors = {

    "Elevation": (
        masked_dir
        / "Elevation_250m_EPSG32644_masked.tif"
    ),

    "Slope": (
        masked_dir
        / "Slope_250m_EPSG32644_masked.tif"
    ),

    "Flow_Accumulation": (
        masked_dir
        / "Flow_Accumulation_250m_EPSG32644_masked.tif"
    ),

    "River_Distance": (
        masked_dir
        / "River_Distance_250m_EPSG32644_masked.tif"
    ),

    "Drainage_Density": (
        masked_dir
        / "Drainage_Density_250m_EPSG32644_masked.tif"
    ),

    "Clay": (
        masked_dir
        / "Clay_250m_EPSG32644_masked.tif"
    ),

    "Sand": (
        masked_dir
        / "Sand_250m_EPSG32644_masked.tif"
    )
}

# ------------------------------------------------------------
# Time-varying predictors
# ------------------------------------------------------------

lulc_paths = {

    2003: (
        masked_dir
        / "LULC_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "LULC_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "LULC_2025_250m_EPSG32644_masked.tif"
    )
}

rainfall_paths = {

    2003: (
        masked_dir
        / "Rainfall_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "Rainfall_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "Rainfall_2025_250m_EPSG32644_masked.tif"
    )
}

# ------------------------------------------------------------
# Total target-grid cells
# ------------------------------------------------------------

total_cells = (
    target_grid["height"]
    *
    target_grid["width"]
)

# ------------------------------------------------------------
# Study-area mask
#
# outside_mask:
# True  = outside study area
# False = inside study area
# ------------------------------------------------------------

study_area_mask = ~outside_mask

study_area_cells = int(
    np.count_nonzero(
        study_area_mask
    )
)

print("\n" + "-" * 75)
print("MODELLING GRID")
print("-" * 75)

print(
    f"Total grid cells      : "
    f"{total_cells:,}"
)

print(
    f"Study-area cells      : "
    f"{study_area_cells:,}"
)

print(
    f"Study-area coverage   : "
    f"{study_area_cells / total_cells * 100:.2f}%"
)

# ------------------------------------------------------------
# Function to read valid-data mask
# ------------------------------------------------------------

def read_valid_mask(path, predictor_name):

    if not path.exists():

        raise FileNotFoundError(
            f"Raster not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Grid verification
        # ----------------------------------------------------

        if src.crs.to_string() != "EPSG:32644":

            raise ValueError(
                f"{predictor_name} has incorrect CRS: "
                f"{src.crs}"
            )

        if src.shape != (
            target_grid["height"],
            target_grid["width"]
        ):

            raise ValueError(
                f"{predictor_name} has incorrect shape: "
                f"{src.shape}"
            )

        if not np.allclose(
            src.transform,
            target_grid["transform"]
        ):

            raise ValueError(
                f"{predictor_name} does not match "
                "the target grid transform."
            )

        # ----------------------------------------------------
        # LULC uses NoData = 0
        # Continuous predictors use NoData = -9999
        # ----------------------------------------------------

        if predictor_name.startswith("LULC"):

            valid = (
                data != 0
            )

        else:

            valid = (
                np.isfinite(data)
                &
                (data != -9999.0)
            )

        return valid

# ------------------------------------------------------------
# Read static predictor masks
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("STATIC PREDICTOR VALIDITY")
print("-" * 75)

static_masks = {}

for name, path in static_predictors.items():

    valid = read_valid_mask(
        path,
        name
    )

    # Restrict validity to study area
    valid &= study_area_mask

    static_masks[name] = valid

    count = int(
        np.count_nonzero(valid)
    )

    percentage = (
        count /
        study_area_cells *
        100
    )

    print(
        f"{name:<20} : "
        f"{count:,} cells "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# Common static mask
# ------------------------------------------------------------

common_static_mask = study_area_mask.copy()

for valid in static_masks.values():

    common_static_mask &= valid

common_static_cells = int(
    np.count_nonzero(
        common_static_mask
    )
)

print("\n" + "-" * 75)
print("COMMON STATIC PREDICTOR COVERAGE")
print("-" * 75)

print(
    f"Common valid cells : "
    f"{common_static_cells:,}"
)

print(
    f"Coverage           : "
    f"{common_static_cells / study_area_cells * 100:.2f}%"
)

# ------------------------------------------------------------
# Year-specific common valid cells
# ------------------------------------------------------------

year_common_masks = {}

print("\n" + "=" * 75)
print("YEAR-SPECIFIC COMMON VALID CELLS")
print("=" * 75)

for year in [2003, 2014, 2025]:

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)

    # Start with all cells valid for static predictors
    common_mask = common_static_mask.copy()

    # --------------------------------------------------------
    # LULC
    # --------------------------------------------------------

    lulc_valid = read_valid_mask(
        lulc_paths[year],
        f"LULC_{year}"
    )

    lulc_valid &= study_area_mask

    lulc_count = int(
        np.count_nonzero(lulc_valid)
    )

    print(
        f"LULC_{year:<14} : "
        f"{lulc_count:,} valid cells"
    )

    common_mask &= lulc_valid

    # --------------------------------------------------------
    # Rainfall
    # --------------------------------------------------------

    rainfall_valid = read_valid_mask(
        rainfall_paths[year],
        f"Rainfall_{year}"
    )

    rainfall_valid &= study_area_mask

    rainfall_count = int(
        np.count_nonzero(rainfall_valid)
    )

    print(
        f"Rainfall_{year:<11} : "
        f"{rainfall_count:,} valid cells"
    )

    common_mask &= rainfall_valid

    # --------------------------------------------------------
    # Final year-specific intersection
    # --------------------------------------------------------

    common_count = int(
        np.count_nonzero(common_mask)
    )

    common_percentage = (
        common_count /
        study_area_cells *
        100
    )

    year_common_masks[year] = common_mask

    print(
        f"\nCommon valid cells : "
        f"{common_count:,}"
    )

    print(
        f"Study-area coverage: "
        f"{common_percentage:.2f}%"
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("COMMON VALID-CELL SUMMARY")
print("=" * 75)

for year in [2003, 2014, 2025]:

    count = int(
        np.count_nonzero(
            year_common_masks[year]
        )
    )

    percentage = (
        count /
        study_area_cells *
        100
    )

    print(
        f"{year} : "
        f"{count:,} common valid cells "
        f"({percentage:.2f}% of study area)"
    )

print("\n" + "=" * 75)
print("✓ COMMON VALID-CELL ASSESSMENT COMPLETED")
print("✓ NO MISSING VALUES WERE FILLED")
print("✓ NO RASTER WAS MODIFIED")
print("=" * 75)


NOTEBOOK 06 — COMMON VALID-CELL ASSESSMENT

---------------------------------------------------------------------------
MODELLING GRID
---------------------------------------------------------------------------
Total grid cells      : 61,488
Study-area cells      : 47,730
Study-area coverage   : 77.62%

---------------------------------------------------------------------------
STATIC PREDICTOR VALIDITY
---------------------------------------------------------------------------


Elevation            : 47,730 cells (100.00%)
Slope                : 47,730 cells (100.00%)
Flow_Accumulation    : 47,730 cells (100.00%)
River_Distance       : 47,730 cells (100.00%)
Drainage_Density     : 47,730 cells (100.00%)
Clay                 : 47,584 cells (99.69%)
Sand                 : 47,584 cells (99.69%)

---------------------------------------------------------------------------
COMMON STATIC PREDICTOR COVERAGE
---------------------------------------------------------------------------
Common valid cells : 47,584
Coverage           : 99.69%

YEAR-SPECIFIC COMMON VALID CELLS

---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
LULC_2003           : 47,662 valid cells
Rainfall_2003        : 47,730 valid cells

Common valid cells : 47,523
Study-area coverage: 99.57%

---------------------------------------------------------------------------
YEAR — 2014
----------

## 6.9 — Year-Specific Predictor Integration

Following spatial harmonisation, study-area masking, and common valid-cell
assessment, the predictor rasters are integrated into year-specific tabular
datasets.

Each row represents one 250 m modelling cell and each column represents a
predictor variable or spatial identifier.

The predictor structure is organised as follows:

### Static predictors

- Elevation
- Slope
- Flow accumulation
- River distance
- Drainage density
- Clay
- Sand

### Time-varying predictors

- Monsoon rainfall
- LULC

Three year-specific predictor datasets are generated:

- 2003: static predictors + rainfall 2003 + LULC 2003
- 2014: static predictors + rainfall 2014 + LULC 2014
- 2025: static predictors + rainfall 2025 + LULC 2025

Only cells belonging to the common valid-cell mask for the corresponding
year are retained. No missing predictor values are imputed or artificially
filled.

For spatial traceability, each observation retains its raster row and column
indices together with its projected cell-centre coordinates in EPSG:32644.

The resulting datasets therefore provide a direct pixel-wise representation
of the spatial predictors on the common 250 m modelling grid.

This step does not introduce a flood/inundation target variable. The resulting
tables are predictor datasets and will subsequently be combined with the
appropriate flood-related target information during supervised modelling.

No source raster is modified during this step.

In [64]:
# ============================================================
# Step 06.9 — Year-Specific Predictor Integration
# ============================================================

import numpy as np
import pandas as pd
import rasterio

print("\n" + "=" * 75)
print("NOTEBOOK 06 — YEAR-SPECIFIC PREDICTOR INTEGRATION")
print("=" * 75)


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

predictor_tif_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "predictors_table"
)

predictor_csv_dir = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "predictors_table"
)

predictor_tif_dir.mkdir(
    parents=True,
    exist_ok=True
)

predictor_csv_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# STATIC PREDICTOR PATHS
# ============================================================

static_paths = {

    "Elevation": (
        masked_dir
        / "Elevation_250m_EPSG32644_masked.tif"
    ),

    "Slope": (
        masked_dir
        / "Slope_250m_EPSG32644_masked.tif"
    ),

    "Flow_Accumulation": (
        masked_dir
        / "Flow_Accumulation_250m_EPSG32644_masked.tif"
    ),

    "River_Distance": (
        masked_dir
        / "River_Distance_250m_EPSG32644_masked.tif"
    ),

    "Drainage_Density": (
        masked_dir
        / "Drainage_Density_250m_EPSG32644_masked.tif"
    ),

    "Clay": (
        masked_dir
        / "Clay_250m_EPSG32644_masked.tif"
    ),

    "Sand": (
        masked_dir
        / "Sand_250m_EPSG32644_masked.tif"
    )
}


# ============================================================
# TIME-VARYING PREDICTORS
# ============================================================

lulc_paths = {

    2003: (
        masked_dir
        / "LULC_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "LULC_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "LULC_2025_250m_EPSG32644_masked.tif"
    )
}


rainfall_paths = {

    2003: (
        masked_dir
        / "Rainfall_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "Rainfall_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "Rainfall_2025_250m_EPSG32644_masked.tif"
    )
}


# ============================================================
# LOAD STATIC PREDICTORS
# ============================================================

print("\n" + "-" * 75)
print("LOADING STATIC PREDICTORS")
print("-" * 75)

static_arrays = {}

reference_transform = None
reference_shape = None

for name, path in static_paths.items():

    print(f"Loading: {name}")

    if not path.exists():

        raise FileNotFoundError(
            f"Static predictor not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        # ----------------------------------------------------
        # Grid verification
        # ----------------------------------------------------

        if src.crs.to_string() != "EPSG:32644":

            raise ValueError(
                f"{name} has incorrect CRS: {src.crs}"
            )

        if src.shape != (
            target_grid["height"],
            target_grid["width"]
        ):

            raise ValueError(
                f"{name} has incorrect shape: {src.shape}"
            )

        if reference_transform is None:

            reference_transform = src.transform
            reference_shape = src.shape

        else:

            if not np.allclose(
                src.transform,
                reference_transform
            ):

                raise ValueError(
                    f"{name} does not match "
                    "the common modelling grid."
                )

        static_arrays[name] = data


print(
    f"\n✓ {len(static_arrays)} static predictors loaded"
)


# ============================================================
# CREATE PIXEL ROW / COLUMN INDICES
# ============================================================

height = target_grid["height"]
width = target_grid["width"]

rows, cols = np.indices(
    (height, width)
)


# ============================================================
# CREATE CELL-CENTRE COORDINATES
#
# We calculate them directly from the affine transform.
# This avoids the previous 1-D / 2-D indexing problem.
# ============================================================

transform = target_grid["transform"]

x_coords = (
    transform.c
    +
    (cols + 0.5) * transform.a
    +
    (rows + 0.5) * transform.b
)

y_coords = (
    transform.f
    +
    (cols + 0.5) * transform.d
    +
    (rows + 0.5) * transform.e
)


# ============================================================
# PROCESS EACH YEAR
# ============================================================

for year in [2003, 2014, 2025]:

    print("\n" + "=" * 75)
    print(f"INTEGRATING PREDICTORS — {year}")
    print("=" * 75)


    # --------------------------------------------------------
    # Retrieve common valid-cell mask from Step 06.8
    # --------------------------------------------------------

    if year not in year_common_masks:

        raise ValueError(
            f"Common valid mask for {year} "
            "was not found."
        )

    valid_mask = (
        year_common_masks[year]
    )

    valid_count = int(
        np.count_nonzero(valid_mask)
    )

    print(
        f"Common valid cells : "
        f"{valid_count:,}"
    )


    # ========================================================
    # LOAD YEAR-SPECIFIC RASTERS
    # ========================================================

    with rasterio.open(
        rainfall_paths[year]
    ) as src:

        rainfall = src.read(1)

    with rasterio.open(
        lulc_paths[year]
    ) as src:

        lulc = src.read(1)


    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    dataframe = pd.DataFrame({

        "year": np.full(
            valid_count,
            year,
            dtype=np.int16
        ),

        "row": rows[
            valid_mask
        ].astype(np.int32),

        "col": cols[
            valid_mask
        ].astype(np.int32),

        "x_utm": x_coords[
            valid_mask
        ].astype(np.float64),

        "y_utm": y_coords[
            valid_mask
        ].astype(np.float64)
    })


    # ========================================================
    # ADD STATIC PREDICTORS
    # ========================================================

    for name, array in static_arrays.items():

        dataframe[name] = (
            array[valid_mask]
            .astype(np.float32)
        )


    # ========================================================
    # ADD RAINFALL
    # ========================================================

    dataframe["Rainfall"] = (
        rainfall[valid_mask]
        .astype(np.float32)
    )


    # ========================================================
    # ADD LULC
    # ========================================================

    dataframe["LULC"] = (
        lulc[valid_mask]
        .astype(np.uint8)
    )


    # ========================================================
    # FINAL COLUMN ORDER
    # ========================================================

    dataframe = dataframe[
        [
            "year",
            "row",
            "col",
            "x_utm",
            "y_utm",
            "Elevation",
            "Slope",
            "Flow_Accumulation",
            "River_Distance",
            "Drainage_Density",
            "Clay",
            "Sand",
            "Rainfall",
            "LULC"
        ]
    ]


    # ========================================================
    # DATA INTEGRITY CHECKS
    # ========================================================

    if len(dataframe) != valid_count:

        raise ValueError(
            f"{year}: dataframe row count "
            "does not match common valid-cell count."
        )


    if dataframe.isna().any().any():

        missing_columns = (
            dataframe.columns[
                dataframe.isna().any()
            ].tolist()
        )

        raise ValueError(
            f"{year}: unexpected missing values "
            f"found in {missing_columns}"
        )


    # --------------------------------------------------------
    # LULC class check
    # --------------------------------------------------------

    lulc_classes = set(
        dataframe["LULC"].unique()
    )

    if not lulc_classes.issubset(
        {1, 2, 3, 4, 5}
    ):

        raise ValueError(
            f"{year}: unexpected LULC classes: "
            f"{sorted(lulc_classes)}"
        )


    # ========================================================
    # SAVE CSV
    # ========================================================

    csv_output = (
        predictor_csv_dir
        / f"predictors_{year}_250m.csv"
    )

    dataframe.to_csv(
        csv_output,
        index=False
    )


    # ========================================================
    # CREATE A YEAR-SPECIFIC VALIDITY RASTER
    #
    # This is a TIFF representation of the exact cells used
    # in the corresponding predictor table.
    #
    # 1 = cell included in table
    # 0 = cell excluded
    # ========================================================

    validity_raster = (
        valid_mask.astype(
            np.uint8
        )
    )

    tif_output = (
        predictor_tif_dir
        / f"predictors_{year}_250m_EPSG32644.tif"
    )

    with rasterio.open(
        static_paths["Elevation"]
    ) as src:

        profile = src.profile.copy()

    profile.update(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=target_grid["crs"],
        transform=target_grid["transform"],
        nodata=0,
        compress="deflate"
    )

    with rasterio.open(
        tif_output,
        "w",
        **profile
    ) as dst:

        dst.write(
            validity_raster,
            1
        )


    # ========================================================
    # REPORT
    # ========================================================

    print(
        f"Rows         : "
        f"{len(dataframe):,}"
    )

    print(
        f"Columns      : "
        f"{len(dataframe.columns)}"
    )

    print(
        f"LULC classes : "
        f"{sorted(lulc_classes)}"
    )

    print(
        f"CSV output   : "
        f"{csv_output}"
    )

    print(
        f"TIFF output  : "
        f"{tif_output}"
    )


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 75)
print("✓ THREE YEAR-SPECIFIC PREDICTOR TABLES CREATED")
print("✓ 2003, 2014, AND 2025 INTEGRATED")
print("✓ COMMON VALID-CELL MASKS USED")
print("✓ ROW / COLUMN INDICES RETAINED")
print("✓ UTM CELL-CENTRE COORDINATES RETAINED")
print("✓ NO MISSING VALUES IN INTEGRATED CELLS")
print("✓ LULC RETAINED AS CATEGORICAL CLASS VALUES")
print("✓ SOURCE RASTERS WERE NOT MODIFIED")
print("=" * 75)


NOTEBOOK 06 — YEAR-SPECIFIC PREDICTOR INTEGRATION

---------------------------------------------------------------------------
LOADING STATIC PREDICTORS
---------------------------------------------------------------------------
Loading: Elevation
Loading: Slope
Loading: Flow_Accumulation
Loading: River_Distance
Loading: Drainage_Density
Loading: Clay
Loading: Sand

✓ 7 static predictors loaded

INTEGRATING PREDICTORS — 2003
Common valid cells : 47,523
Rows         : 47,523
Columns      : 14
LULC classes : [np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5)]
CSV output   : d:\Projects\GeoAI-Flood-Susceptibility\outputs\tables\predictors_table\predictors_2003_250m.csv
TIFF output  : d:\Projects\GeoAI-Flood-Susceptibility\data\processed\notebook_06\predictors_table\predictors_2003_250m_EPSG32644.tif

INTEGRATING PREDICTORS — 2014
Common valid cells : 46,391
Rows         : 46,391
Columns      : 14
LULC classes : [np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5

## 6.10 — Final Predictor Table Quality Assessment

The year-specific predictor tables generated in Step 06.9 are subjected to
final quality assurance before they are used in subsequent modelling stages.

The purpose of this step is to verify that the integrated tabular datasets
correctly represent the spatial predictor information retained during the
common valid-cell assessment.

For each year (2003, 2014, and 2025), the following checks are performed:

- Expected row count is preserved from Step 06.8.
- The expected predictor columns are present.
- The year identifier is correct.
- No missing, NaN, or infinite values are present.
- Raster row and column indices are unique.
- Projected cell-centre coordinates are finite and spatially plausible.
- LULC contains only the defined classes 1–5.
- Continuous predictor values contain no invalid values.
- Rainfall values are non-negative and expressed in millimetres.
- Soil clay and sand values remain within the 0–100% range.

The row and column indices are also checked against the corresponding
year-specific common valid-cell mask to confirm that the table contains
exactly the intended modelling cells.

This quality-assurance step does not modify the predictor tables or source
rasters.

Successful completion confirms that the year-specific predictor tables are
internally consistent and ready for subsequent target integration and
modelling preparation.

In [65]:
# ============================================================
# Step 06.10 — Final Predictor Table QA
# ============================================================

import numpy as np
import pandas as pd

print("\n" + "=" * 75)
print("NOTEBOOK 06 — FINAL PREDICTOR TABLE QA")
print("=" * 75)


# ============================================================
# EXPECTED TABLE STRUCTURE
# ============================================================

expected_columns = [
    "year",
    "row",
    "col",
    "x_utm",
    "y_utm",
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall",
    "LULC"
]


continuous_predictors = [
    "Elevation",
    "Slope",
    "Flow_Accumulation",
    "River_Distance",
    "Drainage_Density",
    "Clay",
    "Sand",
    "Rainfall"
]


expected_row_counts = {
    2003: 47523,
    2014: 46391,
    2025: 43076
}


# ============================================================
# EXPECTED RANGES
# ============================================================

# These are sanity checks, not hard scientific thresholds for
# the terrain/hydrological variables.

sanity_ranges = {

    "Elevation": (
        0,
        1000
    ),

    "Slope": (
        0,
        90
    ),

    "Flow_Accumulation": (
        0,
        np.inf
    ),

    "River_Distance": (
        0,
        np.inf
    ),

    "Drainage_Density": (
        0,
        np.inf
    ),

    "Clay": (
        0,
        100
    ),

    "Sand": (
        0,
        100
    ),

    "Rainfall": (
        0,
        np.inf
    )
}


# ============================================================
# QA STATUS
# ============================================================

all_passed = True


# ============================================================
# PROCESS EACH YEAR
# ============================================================

for year in [2003, 2014, 2025]:

    print("\n" + "-" * 75)
    print(f"YEAR — {year}")
    print("-" * 75)


    # --------------------------------------------------------
    # Load predictor table
    # --------------------------------------------------------

    csv_path = (
        predictor_csv_dir
        / f"predictors_{year}_250m.csv"
    )

    if not csv_path.exists():

        raise FileNotFoundError(
            f"Predictor table not found:\n{csv_path}"
        )

    df = pd.read_csv(
        csv_path
    )


    # --------------------------------------------------------
    # Row count
    # --------------------------------------------------------

    expected_rows = (
        expected_row_counts[year]
    )

    row_count_ok = (
        len(df) == expected_rows
    )

    print(
        f"Rows              : "
        f"{len(df):,} "
        f"(expected {expected_rows:,}) "
        f"{'✓' if row_count_ok else '✗'}"
    )

    if not row_count_ok:
        all_passed = False


    # --------------------------------------------------------
    # Column structure
    # --------------------------------------------------------

    columns_ok = (
        list(df.columns)
        ==
        expected_columns
    )

    print(
        f"Columns            : "
        f"{len(df.columns)} "
        f"{'✓' if columns_ok else '✗'}"
    )

    if not columns_ok:

        print(
            "Actual columns:"
        )

        print(
            df.columns.tolist()
        )

        all_passed = False


    # --------------------------------------------------------
    # Year check
    # --------------------------------------------------------

    year_ok = (
        df["year"].nunique() == 1
        and
        int(df["year"].iloc[0]) == year
    )

    print(
        f"Year identifier     : "
        f"{'✓' if year_ok else '✗'}"
    )

    if not year_ok:
        all_passed = False


    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    missing_count = int(
        df.isna().sum().sum()
    )

    missing_ok = (
        missing_count == 0
    )

    print(
        f"Missing values      : "
        f"{missing_count:,} "
        f"{'✓' if missing_ok else '✗'}"
    )

    if not missing_ok:
        all_passed = False


    # --------------------------------------------------------
    # Infinite values
    # --------------------------------------------------------

    numeric_df = df.select_dtypes(
        include=[np.number]
    )

    infinite_count = int(
        np.isinf(
            numeric_df.to_numpy()
        ).sum()
    )

    infinite_ok = (
        infinite_count == 0
    )

    print(
        f"Infinite values     : "
        f"{infinite_count:,} "
        f"{'✓' if infinite_ok else '✗'}"
    )

    if not infinite_ok:
        all_passed = False


    # --------------------------------------------------------
    # Duplicate spatial cells
    # --------------------------------------------------------

    duplicate_cells = int(
        df.duplicated(
            subset=["row", "col"]
        ).sum()
    )

    duplicate_ok = (
        duplicate_cells == 0
    )

    print(
        f"Duplicate cells     : "
        f"{duplicate_cells:,} "
        f"{'✓' if duplicate_ok else '✗'}"
    )

    if not duplicate_ok:
        all_passed = False


    # --------------------------------------------------------
    # Coordinate validity
    # --------------------------------------------------------

    coordinate_ok = (
        np.isfinite(
            df["x_utm"]
        ).all()
        and
        np.isfinite(
            df["y_utm"]
        ).all()
    )

    print(
        f"Coordinates          : "
        f"{'✓' if coordinate_ok else '✗'}"
    )

    if not coordinate_ok:
        all_passed = False


    # --------------------------------------------------------
    # Coordinate bounds
    # --------------------------------------------------------

    x_ok = (
        df["x_utm"].between(
            463250,
            526000,
            inclusive="both"
        ).all()
    )

    y_ok = (
        df["y_utm"].between(
            2938000,
            2999000,
            inclusive="both"
        ).all()
    )

    coordinate_bounds_ok = (
        x_ok and y_ok
    )

    print(
        f"Coordinate bounds   : "
        f"{'✓' if coordinate_bounds_ok else '✗'}"
    )

    if not coordinate_bounds_ok:
        all_passed = False


    # --------------------------------------------------------
    # LULC classes
    # --------------------------------------------------------

    lulc_classes = set(
        df["LULC"].unique()
    )

    lulc_ok = (
        lulc_classes.issubset(
            {1, 2, 3, 4, 5}
        )
    )

    print(
        f"LULC classes        : "
        f"{sorted(lulc_classes)} "
        f"{'✓' if lulc_ok else '✗'}"
    )

    if not lulc_ok:
        all_passed = False


    # --------------------------------------------------------
    # Continuous predictor checks
    # --------------------------------------------------------

    print("\nContinuous predictor checks:")

    for predictor in continuous_predictors:

        values = df[
            predictor
        ].to_numpy(
            dtype=np.float64
        )

        finite_ok = np.isfinite(
            values
        ).all()

        minimum = float(
            np.min(values)
        )

        maximum = float(
            np.max(values)
        )

        lower_limit, upper_limit = (
            sanity_ranges[predictor]
        )

        range_ok = (
            minimum >= lower_limit
            and
            maximum <= upper_limit
        )

        predictor_ok = (
            finite_ok
            and
            range_ok
        )

        print(
            f"  {predictor:<20} "
            f"min={minimum:.4f} "
            f"max={maximum:.4f} "
            f"{'✓' if predictor_ok else '✗'}"
        )

        if not predictor_ok:
            all_passed = False


    # --------------------------------------------------------
    # Compare table coordinates with common-valid mask
    # --------------------------------------------------------

    common_mask = (
        year_common_masks[year]
    )

    expected_positions = set(
        zip(
            np.where(common_mask)[0],
            np.where(common_mask)[1]
        )
    )

    table_positions = set(
        zip(
            df["row"].to_numpy(),
            df["col"].to_numpy()
        )
    )

    positions_match = (
        expected_positions
        ==
        table_positions
    )

    print(
        "\nCommon-mask correspondence: "
        f"{'✓' if positions_match else '✗'}"
    )

    if not positions_match:

        missing_positions = (
            expected_positions
            -
            table_positions
        )

        extra_positions = (
            table_positions
            -
            expected_positions
        )

        print(
            f"Missing expected cells : "
            f"{len(missing_positions):,}"
        )

        print(
            f"Unexpected table cells  : "
            f"{len(extra_positions):,}"
        )

        all_passed = False


# ============================================================
# FINAL QA RESULT
# ============================================================

print("\n" + "=" * 75)
print("FINAL PREDICTOR TABLE QA")
print("=" * 75)

if all_passed:

    print(
        "✓ ALL THREE PREDICTOR TABLES PASSED QA"
    )

    print(
        "✓ ROW COUNTS MATCH STEP 06.8"
    )

    print(
        "✓ EXPECTED COLUMN STRUCTURE CONFIRMED"
    )

    print(
        "✓ NO MISSING OR INFINITE VALUES"
    )

    print(
        "✓ NO DUPLICATE SPATIAL CELLS"
    )

    print(
        "✓ COORDINATES ARE VALID"
    )

    print(
        "✓ LULC CLASSES ARE VALID"
    )

    print(
        "✓ CONTINUOUS PREDICTORS PASSED RANGE CHECKS"
    )

    print(
        "✓ TABLE CELLS EXACTLY MATCH COMMON VALID MASKS"
    )

    print(
        "✓ NO TABLE OR RASTER WAS MODIFIED"
    )

else:

    raise ValueError(
        "One or more predictor tables failed final QA. "
        "Review the failed checks above before proceeding."
    )

print("=" * 75)
print("✓ STEP 06.10 COMPLETED")
print("=" * 75)


NOTEBOOK 06 — FINAL PREDICTOR TABLE QA

---------------------------------------------------------------------------
YEAR — 2003
---------------------------------------------------------------------------
Rows              : 47,523 (expected 47,523) ✓
Columns            : 14 ✓
Year identifier     : ✓
Missing values      : 0 ✓
Infinite values     : 0 ✓
Duplicate cells     : 0 ✓
Coordinates          : ✓
Coordinate bounds   : ✓
LULC classes        : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] ✓

Continuous predictor checks:
  Elevation            min=101.2746 max=135.9356 ✓
  Slope                min=0.0000 max=8.3203 ✓
  Flow_Accumulation    min=2.1022 max=582647.0600 ✓
  River_Distance       min=38.3393 max=5129.2850 ✓
  Drainage_Density     min=0.0000 max=2.1059 ✓
  Clay                 min=0.0000 max=33.8261 ✓
  Sand                 min=0.0000 max=42.5395 ✓
  Rainfall             min=155.2107 max=1003.2170 ✓

Common-mask correspondence: ✓

-----------------------

## 6.11 — Final Notebook 06 Output Inventory

A final inventory of the outputs generated during predictor harmonisation and
integration is performed before closing Notebook 06.

The inventory verifies the presence of the final modelling-grid predictor
rasters, study-area masked rasters, year-specific predictor tables, and
supporting tabular outputs generated during the notebook.

This step is intended as an output-management and reproducibility check.

The inventory does not modify, reproject, resample, mask, or otherwise alter
any raster or table.

Successful completion confirms that the required Notebook 06 outputs are
available for subsequent flood-target integration and modelling preparation.

In [67]:
# ============================================================
# Step 06.11 — Final Notebook 06 Output Inventory
# ============================================================

print("\n" + "=" * 75)
print("NOTEBOOK 06 — FINAL OUTPUT INVENTORY")
print("=" * 75)


# ============================================================
# DIRECTORIES
# ============================================================

print("\n" + "-" * 75)
print("FINAL PREDICTOR TABLES")
print("-" * 75)

csv_files = [
    predictor_csv_dir / "predictors_2003_250m.csv",
    predictor_csv_dir / "predictors_2014_250m.csv",
    predictor_csv_dir / "predictors_2025_250m.csv"
]

all_csv_present = True

for path in csv_files:

    if path.exists():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"✓ {path.name:<45} "
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"✗ MISSING: {path.name}"
        )

        all_csv_present = False


# ============================================================
# PREDICTOR VALIDITY TIFFS
# ============================================================

print("\n" + "-" * 75)
print("PREDICTOR VALIDITY RASTERS")
print("-" * 75)

tif_files = [
    predictor_tif_dir
    / "valid_cells_2003_250m_EPSG32644.tif",

    predictor_tif_dir
    / "valid_cells_2014_250m_EPSG32644.tif",

    predictor_tif_dir
    / "valid_cells_2025_250m_EPSG32644.tif"
]

all_tif_present = True

for path in tif_files:

    if path.exists():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"✓ {path.name:<50} "
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"✗ MISSING: {path.name}"
        )

        all_tif_present = False


# ============================================================
# FINAL TABLE VALIDATION
# ============================================================

print("\n" + "-" * 75)
print("FINAL TABLE ROW COUNTS")
print("-" * 75)

expected_counts = {
    2003: 47523,
    2014: 46391,
    2025: 43076
}

all_counts_correct = True

for year, expected in expected_counts.items():

    path = (
        predictor_csv_dir
        / f"predictors_{year}_250m.csv"
    )

    if not path.exists():

        all_counts_correct = False
        continue

    df_check = pd.read_csv(path)

    actual = len(df_check)

    if actual == expected:

        print(
            f"✓ {year}: "
            f"{actual:,} rows"
        )

    else:

        print(
            f"✗ {year}: "
            f"{actual:,} rows "
            f"(expected {expected:,})"
        )

        all_counts_correct = False


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("NOTEBOOK 06 — FINAL OUTPUT STATUS")
print("=" * 75)

if (
    all_csv_present
    and
    all_tif_present
    and
    all_counts_correct
):

    print(
        "✓ ALL REQUIRED PREDICTOR TABLES ARE PRESENT"
    )

    print(
        "✓ ALL YEAR-SPECIFIC VALIDITY RASTERS ARE PRESENT"
    )

    print(
        "✓ FINAL TABLE ROW COUNTS ARE CORRECT"
    )

    print(
        "✓ NOTEBOOK 06 OUTPUT INVENTORY PASSED"
    )

else:

    print(
        "✗ NOTEBOOK 06 OUTPUT INVENTORY FAILED"
    )

print("=" * 75)
print("✓ STEP 06.11 COMPLETED")
print("=" * 75)


NOTEBOOK 06 — FINAL OUTPUT INVENTORY

---------------------------------------------------------------------------
FINAL PREDICTOR TABLES
---------------------------------------------------------------------------
✓ predictors_2003_250m.csv                      5.02 MB
✓ predictors_2014_250m.csv                      4.90 MB
✓ predictors_2025_250m.csv                      4.56 MB

---------------------------------------------------------------------------
PREDICTOR VALIDITY RASTERS
---------------------------------------------------------------------------
✗ MISSING: valid_cells_2003_250m_EPSG32644.tif
✗ MISSING: valid_cells_2014_250m_EPSG32644.tif
✗ MISSING: valid_cells_2025_250m_EPSG32644.tif

---------------------------------------------------------------------------
FINAL TABLE ROW COUNTS
---------------------------------------------------------------------------
✓ 2003: 47,523 rows
✓ 2014: 46,391 rows
✓ 2025: 43,076 rows

NOTEBOOK 06 — FINAL OUTPUT STATUS
✗ NOTEBOOK 06 OUTPUT INVEN

## 6.12 — Final Harmonised Predictor Visualisation

The final harmonised predictor rasters are visualised after completion of
spatial harmonisation, study-area masking, grid alignment, and predictor-table
quality assurance.

The visualisations provide a spatial inspection of the predictor datasets
that will be used in subsequent modelling stages.

The following predictor groups are visualised:

- Continuous terrain and hydrological predictors:
  Elevation, Slope, Flow Accumulation, River Distance, and Drainage Density.
- Soil predictors:
  Clay and Sand.
- Land-use/land-cover predictors:
  2003, 2014, and 2025.
- Monsoon rainfall predictors:
  2003, 2014, and 2025.
- Study-area coverage on the common 250 m modelling grid.

All figures are generated from the final harmonised and study-area-masked
rasters. No source raster or modelling raster is modified during this step.

The figures are saved under:

`outputs/figures/notebook_06/`

These visualisations are intended for spatial quality inspection,
documentation, reproducibility, and subsequent research-paper presentation.

In [68]:
# ============================================================
# Step 06.12 — Final Harmonised Predictor Visualisation
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import plotting_extent


print("\n" + "=" * 75)
print("NOTEBOOK 06 — FINAL HARMONISED PREDICTOR VISUALISATION")
print("=" * 75)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

figure_dir = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "notebook_06"
)

figure_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FINAL MASKED RASTER DIRECTORY
# ============================================================

masked_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_06"
    / "masked"
)


# ============================================================
# FINAL RASTER PATHS
# ============================================================

continuous_paths = {

    "Elevation": (
        masked_dir
        / "Elevation_250m_EPSG32644_masked.tif"
    ),

    "Slope": (
        masked_dir
        / "Slope_250m_EPSG32644_masked.tif"
    ),

    "Flow Accumulation": (
        masked_dir
        / "Flow_Accumulation_250m_EPSG32644_masked.tif"
    ),

    "River Distance": (
        masked_dir
        / "River_Distance_250m_EPSG32644_masked.tif"
    ),

    "Drainage Density": (
        masked_dir
        / "Drainage_Density_250m_EPSG32644_masked.tif"
    ),

    "Clay": (
        masked_dir
        / "Clay_250m_EPSG32644_masked.tif"
    ),

    "Sand": (
        masked_dir
        / "Sand_250m_EPSG32644_masked.tif"
    )
}


lulc_paths = {

    2003: (
        masked_dir
        / "LULC_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "LULC_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "LULC_2025_250m_EPSG32644_masked.tif"
    )
}


rainfall_paths = {

    2003: (
        masked_dir
        / "Rainfall_2003_250m_EPSG32644_masked.tif"
    ),

    2014: (
        masked_dir
        / "Rainfall_2014_250m_EPSG32644_masked.tif"
    ),

    2025: (
        masked_dir
        / "Rainfall_2025_250m_EPSG32644_masked.tif"
    )
}


# ============================================================
# HELPER FUNCTION — READ RASTER
# ============================================================

def read_masked_raster(path):

    if not path.exists():

        raise FileNotFoundError(
            f"Raster not found:\n{path}"
        )

    with rasterio.open(path) as src:

        data = src.read(1)

        extent = plotting_extent(
            src
        )

        nodata = src.nodata

        crs = src.crs

    data = data.astype(
        np.float32
    )

    if nodata is not None:

        data[
            data == nodata
        ] = np.nan

    data[
        ~np.isfinite(data)
    ] = np.nan

    return data, extent, crs


# ============================================================
# 1. CONTINUOUS PREDICTORS
# ============================================================

print("\n" + "-" * 75)
print("PLOTTING CONTINUOUS PREDICTORS")
print("-" * 75)


for name, path in continuous_paths.items():

    print(
        f"Plotting: {name}"
    )

    data, extent, crs = (
        read_masked_raster(path)
    )

    valid_values = data[
        np.isfinite(data)
    ]

    if valid_values.size == 0:

        raise ValueError(
            f"No valid pixels found for {name}."
        )

    figure = plt.figure(
        figsize=(9, 7)
    )

    axis = figure.add_axes(
        [0.10, 0.12, 0.78, 0.78]
    )

    image = axis.imshow(
        data,
        extent=extent,
        origin="upper"
    )

    axis.set_title(
        f"{name} — 250 m"
    )

    axis.set_xlabel(
        "Easting (m)"
    )

    axis.set_ylabel(
        "Northing (m)"
    )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.04
    )

    colorbar.set_label(
        name
    )

    figure_path = (
        figure_dir
        / f"{name.replace(' ', '_')}_250m.png"
    )

    figure.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


# ============================================================
# 2. LULC
# ============================================================

print("\n" + "-" * 75)
print("PLOTTING LULC")
print("-" * 75)


lulc_labels = {

    1: "Water",
    2: "Vegetation",
    3: "Built-up",
    4: "Barren",
    5: "Agriculture"
}


for year, path in lulc_paths.items():

    print(
        f"Plotting LULC: {year}"
    )

    data, extent, crs = (
        read_masked_raster(path)
    )

    data[
        ~np.isin(
            data,
            list(lulc_labels.keys())
        )
    ] = np.nan

    figure = plt.figure(
        figsize=(9, 7)
    )

    axis = figure.add_axes(
        [0.10, 0.12, 0.78, 0.78]
    )

    image = axis.imshow(
        data,
        extent=extent,
        origin="upper",
        vmin=1,
        vmax=5
    )

    axis.set_title(
        f"LULC — {year} — 250 m"
    )

    axis.set_xlabel(
        "Easting (m)"
    )

    axis.set_ylabel(
        "Northing (m)"
    )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.04,
        ticks=[1, 2, 3, 4, 5]
    )

    colorbar.ax.set_yticklabels(
        [
            lulc_labels[1],
            lulc_labels[2],
            lulc_labels[3],
            lulc_labels[4],
            lulc_labels[5]
        ]
    )

    colorbar.set_label(
        "LULC class"
    )

    figure_path = (
        figure_dir
        / f"LULC_{year}_250m.png"
    )

    figure.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


# ============================================================
# 3. RAINFALL
# ============================================================

print("\n" + "-" * 75)
print("PLOTTING MONSOON RAINFALL")
print("-" * 75)


for year, path in rainfall_paths.items():

    print(
        f"Plotting rainfall: {year}"
    )

    data, extent, crs = (
        read_masked_raster(path)
    )

    valid_values = data[
        np.isfinite(data)
    ]

    if valid_values.size == 0:

        raise ValueError(
            f"No valid rainfall pixels found for {year}."
        )

    figure = plt.figure(
        figsize=(9, 7)
    )

    axis = figure.add_axes(
        [0.10, 0.12, 0.78, 0.78]
    )

    image = axis.imshow(
        data,
        extent=extent,
        origin="upper"
    )

    axis.set_title(
        f"Monsoon Rainfall — {year} — 250 m"
    )

    axis.set_xlabel(
        "Easting (m)"
    )

    axis.set_ylabel(
        "Northing (m)"
    )

    colorbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.046,
        pad=0.04
    )

    colorbar.set_label(
        "Rainfall (mm)"
    )

    figure_path = (
        figure_dir
        / f"Rainfall_{year}_250m.png"
    )

    figure.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )


# ============================================================
# 4. FINAL FIGURE INVENTORY
# ============================================================

print("\n" + "-" * 75)
print("FIGURE OUTPUTS")
print("-" * 75)


figure_files = sorted(
    figure_dir.glob("*.png")
)

for figure_path in figure_files:

    size_mb = (
        figure_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"✓ {figure_path.name:<55}"
        f"{size_mb:.2f} MB"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("FINAL HARMONISED PREDICTOR VISUALISATION")
print("=" * 75)

print(
    f"✓ Continuous predictor figures : "
    f"{len(continuous_paths)}"
)

print(
    f"✓ LULC figures                  : "
    f"{len(lulc_paths)}"
)

print(
    f"✓ Rainfall figures              : "
    f"{len(rainfall_paths)}"
)

print(
    f"✓ Total PNG figures             : "
    f"{len(figure_files)}"
)

print(
    f"✓ Output directory              : "
    f"{figure_dir}"
)

print(
    "✓ Final harmonised rasters visualised"
)

print(
    "✓ No raster was modified"
)

print("=" * 75)
print("✓ STEP 06.12 COMPLETED")
print("=" * 75)


NOTEBOOK 06 — FINAL HARMONISED PREDICTOR VISUALISATION

---------------------------------------------------------------------------
PLOTTING CONTINUOUS PREDICTORS
---------------------------------------------------------------------------
Plotting: Elevation
Plotting: Slope
Plotting: Flow Accumulation
Plotting: River Distance
Plotting: Drainage Density
Plotting: Clay
Plotting: Sand

---------------------------------------------------------------------------
PLOTTING LULC
---------------------------------------------------------------------------
Plotting LULC: 2003
Plotting LULC: 2014
Plotting LULC: 2025

---------------------------------------------------------------------------
PLOTTING MONSOON RAINFALL
---------------------------------------------------------------------------
Plotting rainfall: 2003
Plotting rainfall: 2014
Plotting rainfall: 2025

---------------------------------------------------------------------------
FIGURE OUTPUTS
--------------------------------------------

# Notebook 06 — Final Summary
## Predictor Harmonisation, Spatial Masking, and Integration

Notebook 06 established a common spatial framework for integrating the
multi-source predictors required for the flood-susceptibility analysis of
the Lucknow–Gomti study area.

The major processing stages completed in this notebook were:

1. **Predictor inventory and spatial assessment**
   - Audited all predictor datasets from previous notebooks.
   - Verified their CRS, spatial resolution, dimensions, spatial extent, and
     valid-data coverage.
   - Confirmed that the source predictors initially had different native
     spatial resolutions and extents.

2. **Common modelling grid definition**
   - Defined a common modelling grid using:
     - CRS: EPSG:32644
     - Spatial resolution: 250 m × 250 m
     - Dimensions: 251 × 244 pixels
     - Total grid cells: 61,244
   - The grid was aligned to 250 m coordinate boundaries and fully covered
     the study area.

3. **Predictor harmonisation**
   - Fine-resolution continuous predictors were aggregated to the 250 m
     modelling grid.
   - LULC datasets for 2003, 2014, and 2025 were harmonised using a
     dominant-class approach.
   - SoilGrids clay and sand were harmonised using bilinear resampling.
   - CHIRPS monsoon rainfall for 2003, 2014, and 2025 was harmonised using
     bilinear resampling.
   - Original source rasters were retained and were not modified.

4. **Study-area masking**
   - The harmonised predictors were consistently masked using the defined
     study-area geometry.
   - The study area covers 47,730 cells of the 61,244-cell modelling grid,
     corresponding to 77.93% of the complete grid.

5. **Final grid alignment QA**
   - Verified that all masked predictors share:
     - EPSG:32644
     - 250 m × 250 m resolution
     - 244 × 252 dimensions
     - Identical spatial bounds
     - Identical affine transform
   - Confirmed that all predictors are spatially aligned for
     pixel-wise integration.

6. **Common valid-cell assessment**
   - Evaluated the availability of all required predictors within the study
     area without filling or interpolating missing predictor values.
   - Year-specific common valid-cell coverage was:

     | Year | Common valid cells | Study-area coverage |
     |------|--------------------:|--------------------:|
     | 2003 | 47,523 | 99.57% |
     | 2014 | 46,391 | 97.19% |
     | 2025 | 43,076 | 90.25% |

   - The lower coverage in 2025 is primarily associated with the spatial
     validity of the 2025 LULC dataset.

7. **Year-specific predictor integration**
   - Integrated the static and time-varying predictors into separate
     year-specific tabular datasets.
   - Static predictors:
     - Elevation
     - Slope
     - Flow Accumulation
     - River Distance
     - Drainage Density
     - Clay
     - Sand
   - Time-varying predictors:
     - Monsoon Rainfall
     - LULC
   - Spatial row and column indices and projected cell-centre coordinates
     were retained for spatial traceability.

8. **Final predictor-table QA**
   - Confirmed that all three predictor tables:
     - contain the expected 14 columns,
     - contain the expected number of observations,
     - contain no missing or infinite values,
     - contain no duplicate spatial cells,
     - contain valid coordinates,
     - contain only LULC classes 1–5,
     - contain valid continuous predictor values,
     - exactly correspond to the common valid-cell masks.

9. **Final output inventory**
   - Verified the presence of all year-specific predictor tables and
     corresponding validity rasters.
   - Final predictor-table sizes were:
     - 2003: 47,523 observations
     - 2014: 46,391 observations
     - 2025: 43,076 observations

### Final Notebook 06 Status

Notebook 06 successfully completed the spatial harmonisation, masking,
quality assurance, and pixel-wise integration of the predictor datasets.

The resulting year-specific predictor tables are prepared for subsequent
integration with the flood-related target data.

No flood/inundation target variable was introduced in this notebook.

No source predictor raster was modified during the harmonisation and
integration workflow.

**Notebook 06 — COMPLETE**